# LLM (언어 모델 이해) 학습 노트

> 인증평가(11/17)는 **교안 ipynb 코드를 그대로 출제**하고, 빈칸이나 함수 전체를 직접 타이핑하는 방식입니다.
> 이 노트는 "이론 → Tensor shape 흐름 → 외워서 쓸 코드 → 간단 실습" 순서로 정리했습니다.
> 출처: `1. 언어 모델 이해/pdf/app_sp_slides_20261013.pdf`, `1. 언어 모델 이해/Chapter_1~7_*.ipynb`

## 목차

- [기호 사전 (먼저 읽기)](#기호-사전-먼저-읽기)
- [0. 기본 수학 (코드에 바로 쓰이는 것만)](#0-기본-수학-코드에-바로-쓰이는-것만)
- [1. 벡터 공간 (Chapter 1)](#1-벡터-공간-chapter-1)
- [2. 데이터셋: 토크나이저 · 슬라이딩 윈도우 · 임베딩 (Chapter 2)](#2-데이터셋-토크나이저--슬라이딩-윈도우--임베딩-chapter-2)
- [3. Attention (Chapter 3)](#3-attention-chapter-3)
- [4. GPT 모델 조립 (Chapter 4)](#4-gpt-모델-조립-chapter-4)
- [5. 사전학습 Pretraining (Chapter 5)](#5-사전학습-pretraining-chapter-5)
- [6. 분류 미세 튜닝 · LoRA (Chapter 6)](#6-분류-미세-튜닝--lora-chapter-6)
- [7. 지시 미세 튜닝 · DPO (Chapter 7)](#7-지시-미세-튜닝--dpo-chapter-7)
- [부록. 전체 Tensor shape 한눈에 보기](#부록-전체-tensor-shape-한눈에-보기)
- [부록. 교안 빈칸 전체 정답표 (Chapter 2~7)](#부록-교안-빈칸-전체-정답표-chapter-27)
- [부록. 빈칸 대비 체크리스트](#부록-빈칸-대비-체크리스트)

## 기호 사전 (먼저 읽기)

교안 코드와 이 노트에 나오는 변수 이름/기호를 한곳에 모았습니다. shape를 읽을 때 막히면 여기로 돌아오세요.

| 기호 / 변수명 | 정확한 의미 | 예시 값 |
| --- | --- | --- |
| `b`, `B`, `batch_size` | 한 번에 모델에 넣는 **문장(샘플) 개수** | 2, 8 |
| `T`, `num_tokens`, `seq_len` | 문장 하나에 들어 있는 **토큰 개수** (지금 입력의 실제 길이) | 4, 6 |
| `context_length` | 모델이 처리할 수 있는 **최대** 토큰 개수 (위치 임베딩 표의 행 수). `T ≤ context_length` | 1024, 256 |
| `max_length` | 데이터셋을 자를 때 한 조각의 토큰 수. 교안에서는 보통 `context_length`와 같게 둠 | 4, 256 |
| `stride` | 다음 조각을 자를 때 **몇 칸 이동**하는지 | 1, 4, 128 |
| `vocab_size`, `V` | 토크나이저가 아는 **토큰 종류의 수** (= 토큰 ID의 범위 `0 ~ V-1`) | 50257 |
| `emb_dim`, `d`, `output_dim` | 토큰 하나를 표현하는 **벡터의 길이** (숫자 몇 개로 표현하나) | 768, 256, 3 |
| `d_in` | 어떤 층에 **들어가는** 벡터 하나의 길이 (입력 텐서의 **마지막 차원** 크기) | 3, 768 |
| `d_out` | 그 층에서 **나오는** 벡터 하나의 길이 (출력 텐서의 **마지막 차원** 크기) | 2, 768 |
| `num_heads`, `n_heads`, `H` | Multi-head attention의 head 개수 | 2, 12 |
| `head_dim`, `hd` | head 하나가 맡는 벡터 길이 = `d_out // num_heads` | 64 (=768/12) |
| `n_layers` | TransformerBlock을 몇 개 쌓는지 | 12 |
| logits | softmax 전의 **점수**. 크기 제한 없음 (음수도 가능) | `[2.0, 1.0, 0.1]` |
| `num_classes` | 분류 미세 튜닝에서 고를 클래스 수 = 새 `out_head`의 d_out (Ch6) | 2 (ham/spam) |
| `pad_token_id` | 길이를 맞추려고 채우는 토큰 ID. GPT-2는 `<\|endoftext\|>`를 재사용 | 50256 |
| `ignore_index` | `cross_entropy`가 무시하는 정답 값 (Ch7 SFT) | -100 |
| `rank`, `r` | LoRA에서 A(in, r) · B(r, out) 사이 좁은 통로의 크기 (Ch6) | 16 |
| `alpha`, `α` | LoRA 출력 배율. 실제 배율은 `alpha / rank` (Ch6) | 16 |
| `beta`, `β` | DPO에서 reference로부터 얼마나 벗어날지 조절 (Ch7) | 0.1 |
| policy / reference | DPO의 학습 대상 모델 / 고정된 비교 기준 모델 (Ch7) | |

> **"차원"이라는 말이 두 가지 뜻으로 쓰입니다.**
> - 텐서의 **축 개수** (rank): `(2, 6, 3)`은 3차원 텐서
> - 벡터 하나의 **길이**: "768차원 임베딩" = 숫자 768개짜리 벡터
>
> `d_in`, `d_out`, `emb_dim`의 "d(dimension)"는 **두 번째 뜻(벡터 길이)** 입니다.

shape를 읽는 요령: **맨 마지막 숫자 = 토큰 하나를 표현하는 벡터의 길이**, 그 앞의 숫자들은 "그런 벡터가 몇 개 있는지"를 정리하는 축입니다.

```
(2, 6, 3)  →  문장 2개 × 각 문장의 토큰 6개 × 토큰마다 숫자 3개
               b=2        T=6                 d=3
```

## 0. 기본 수학 (코드에 바로 쓰이는 것만)

### 0-0. 지수함수 · 로그함수 · 미분 (모든 계산의 기초)

LLM 코드에 나오는 거의 모든 수식(softmax, cross entropy, perplexity, 역전파, DPO loss)은 이 세 가지로 이루어져 있습니다. 먼저 어디에 쓰이는지 보고 시작합니다.

| 개념 | 노트에서 쓰이는 곳 |
| --- | --- |
| 지수함수 `e^x` | Softmax (0-4), Temperature, Perplexity `= e^loss` (0-6) |
| 로그함수 `ln x` | Cross Entropy `−ln p` (0-6), `F.log_softmax`, DPO의 로그 확률 (Ch7) |
| 미분 | 역전파 `loss.backward()`, 경사하강법 `optimizer.step()` (0-5, Ch5) |

#### ① 지수함수 `e^x`

**자연상수 e ≈ 2.718**. `e^x`는 "e를 x번 곱한 값"이고, PyTorch에서는 `torch.exp(x)`입니다.

| x | −1 | 0 | 1 | 2 | 8 | 10 |
| --- | --- | --- | --- | --- | --- | --- |
| `e^x` | 0.368 | 1 | 2.718 | 7.389 | 2,981 | 22,026 |

LLM에서 지수함수를 쓰는 이유 (슬라이드 43):

1. **모든 숫자를 양수로 바꿈**: logits는 음수일 수도 있지만 `e^x`는 항상 0보다 큼 → 확률로 만들 수 있음
2. **작은 차이를 크게 벌림**: 단어 점수 A=10, B=8, C=1 → `e^10 ≈ 22026`, `e^8 ≈ 2981`, `e^1 ≈ 2.7`. 점수 차이는 2였지만 7배 이상 차이가 남 → 가장 그럴듯한 단어가 확실히 두드러짐
3. **미분해도 자기 자신**: `d/dx e^x = e^x` → 역전파 계산이 단순해짐

**성질** (로그와 짝을 이룸)

```
e^a · e^b = e^(a+b)        곱하기 → 지수끼리 더하기
e^0 = 1                    그래서 softmax에서 −inf 자리는 e^−inf = 0
e^(−x) = 1 / e^x           sigmoid  σ(z) = 1 / (1 + e^(−z))
```

**오버플로 주의**: `e^1000`은 너무 커서 `inf`가 됨 → softmax 계산 전에 **최댓값을 빼도 결과가 같다**는 성질을 이용 (Ch5 `generate`의 `logits - logits.max(...)`가 이것)

```
softmax([1000, 999]) = softmax([1000−1000, 999−1000]) = softmax([0, −1]) = [0.731, 0.269]
```

In [ ]:
import torch

print(torch.exp(torch.tensor([10., 8., 1.])))     # [22026.4648, 2980.9580, 2.7183]
print(torch.exp(torch.tensor([-1., 0., 1., 2.]))) # [0.3679, 1.0000, 2.7183, 7.3891]

# 오버플로와 해결
z = torch.tensor([1000., 999.])
print(torch.exp(z) / torch.exp(z).sum())               # [nan, nan]  ← inf / inf
z_stable = z - z.max()                                 # [0., -1.]
print(torch.exp(z_stable) / torch.exp(z_stable).sum()) # [0.7311, 0.2689]
print(torch.softmax(z, dim=0))                         # [0.7311, 0.2689]  내장 함수는 내부에서 처리

#### ② 로그함수 `ln x` (= `log x`, 밑이 e)

**지수의 반대**: `e^y = x` 이면 `ln x = y`. "e를 몇 번 곱해야 x가 되나?" PyTorch에서는 `torch.log(x)` (밑이 e인 자연로그).

| x (확률) | 0.01 | 0.1 | 0.5 | 0.9 | 1 |
| --- | --- | --- | --- | --- | --- |
| `ln x` | −4.605 | −2.303 | −0.693 | −0.105 | 0 |
| `−ln x` (loss) | **4.605** | 2.303 | 0.693 | 0.105 | **0** |

- 0 < x < 1 이면 `ln x`는 **음수** → 확률의 로그는 항상 음수. 그래서 Cross Entropy는 앞에 **−**를 붙여 양수로 만듦
- x → 0 이면 `ln x → −∞`: 정답 확률이 0에 가까우면 loss가 폭발적으로 커짐

**성질** (★ 가장 중요)

```
ln(a · b) = ln a + ln b        곱셈 → 덧셈
ln(a / b) = ln a − ln b        나눗셈 → 뺄셈
ln(e^x)  = x                   지수와 로그는 서로 상쇄
ln 1     = 0
```

예: `ln 6 = ln(2 × 3) = ln 2 + ln 3 = 0.693 + 1.099 = 1.792`

LLM에서 로그함수를 쓰는 이유 (슬라이드 43):

1. **곱셈을 덧셈으로 → 언더플로 방지**: 문장 확률 = 토큰 확률들의 **곱**. 토큰 400개가 각각 확률 0.1이면 `0.1^400 = 10^−400` → 컴퓨터는 이렇게 작은 수를 표현할 수 없어서 **0이 됨**. 로그를 취하면 `400 × ln 0.1 = −921.03` → 문제없이 계산
2. **틀렸을 때 강한 피드백**: 정답 확률이 0.9 → 0.5로 떨어질 때 loss는 0.105 → 0.693, 0.01까지 떨어지면 4.605. 많이 틀릴수록 벌점이 급격히 커짐
3. **미분이 깔끔함**: `d/dx ln x = 1/x`

In [ ]:
print(torch.log(torch.tensor([0.01, 0.1, 0.5, 0.9, 1.0])))
# [-4.6052, -2.3026, -0.6931, -0.1054, 0.0000]

# 언더플로: 확률 0.1인 토큰 400개
p = torch.full((400,), 0.1, dtype=torch.float64)
print(torch.prod(p))        # 0.  ← 곱하면 0으로 사라짐 (float64로도 불가능)
print(torch.log(p).sum())   # -921.0340  ← 로그로 더하면 정상 계산

**지수 · 로그가 함께 쓰이는 곳 정리**

| 식 | 의미 |
| --- | --- |
| `softmax(z)_i = e^(z_i) / Σ e^(z_j)` | 지수로 양수화 + 차이 확대 → 합으로 나눠 확률 |
| `log_softmax(z) = z_i − ln Σ e^(z_j)` | softmax 후 log를 한 번에 안정적으로 (DPO의 `F.log_softmax`) |
| `cross_entropy = −ln p(정답)` 평균 | 로그로 벌점 |
| `perplexity = e^(cross_entropy)` | 로그를 다시 지수로 되돌려 "후보 개수"로 해석 |
| `logsigmoid(x) = ln σ(x)` | DPO loss `−F.logsigmoid(β · logits)` |

#### ③ 미분 (기울기)

**미분 = 입력을 아주 조금 바꿨을 때 출력이 얼마나 변하는가** (순간 변화율 = 그래프의 기울기).

```
f(x) = x²  일 때  f'(x) = 2x
x = 3 에서 기울기 = 6  →  x를 0.01 늘리면 f는 약 0.06 늘어남
                            (3.01² = 9.0601, 9 + 0.06 = 9.06 ✔)
```

**자주 쓰는 미분 공식**

| 함수 f(x) | 미분 f'(x) | 쓰이는 곳 |
| --- | --- | --- |
| `c` (상수) | 0 | bias의 상수항 |
| `ax + b` | `a` | 선형층 `z = wx + b` → `∂z/∂w = x` |
| `x²` | `2x` | MSE `(y − ŷ)²` |
| `xⁿ` | `n·xⁿ⁻¹` | `x³ → 3x²` |
| `e^x` | `e^x` | softmax |
| `ln x` | `1/x` | cross entropy |
| `σ(x)` (sigmoid) | `σ(x)(1 − σ(x))` | 슬라이드 49 역전파 예제 |

**편미분 `∂`**: 변수가 여러 개(가중치 수백만 개)일 때, **나머지는 고정하고 하나만** 바꿨을 때의 기울기. `∂L/∂w5` = "w5만 조금 바꾸면 loss가 얼마나 변하나". 모든 가중치에 대한 편미분을 모은 것이 **gradient(기울기 벡터)**.

**연쇄법칙 (Chain Rule)** — 역전파의 핵심

함수가 여러 겹이면 **각 단계의 미분을 곱함**.

```
치타는 사자보다 2배, 사자는 곰보다 2배, 곰은 사람보다 1.5배 빠름
→ 치타/사람 = 2 × 2 × 1.5 = 6배                    (슬라이드 48)

y = (3x + 1)²,  u = 3x + 1 로 두면 y = u²
dy/dx = dy/du × du/dx = 2u × 3 = 2(3x + 1) × 3
x = 1 이면 2 × 4 × 3 = 24
```

신경망: `loss ← softmax ← out_head ← 블록 12 ← ... ← 블록 1 ← 임베딩`. 뒤에서부터 미분을 곱해 가며 **모든 가중치의 기울기를 한 번에** 구하는 것이 역전파(`loss.backward()`).

슬라이드 49의 예 (`o1 = σ(z3)`, `z3 = h1·w5 + h2·w6`, `C = (y − o1)²`):

```
∂C/∂w5 = ∂C/∂o1 × ∂o1/∂z3 × ∂z3/∂w5
       = −2(y − o1) × o1(1 − o1) × h1
       = −0.71 × 0.229 × 0.615 ≈ −0.100
```

> 슬라이드에는 결과가 `≈ −0.01`로 적혀 있지만, 세 수를 실제로 곱하면 `−0.100`입니다. 시험에서 직접 계산이 나오면 곱셈 결과를 따르세요.

**경사하강법 = 미분의 사용처**

기울기는 "loss가 커지는 방향"을 알려 줌 → **반대 방향으로 조금** 움직이면 loss가 줄어듦.

```
w_new = w − α · ∂L/∂w          (α = learning rate)
```

예: `L(w) = (w − 3)²` (w=3일 때 최소), `L'(w) = 2(w − 3)`, 시작 `w = 0`, `α = 0.1`

| step | w | 기울기 2(w−3) | w − 0.1 × 기울기 |
| --- | --- | --- | --- |
| 0 | 0 | −6 | 0.6 |
| 1 | 0.6 | −4.8 | 1.08 |
| 2 | 1.08 | −3.84 | 1.464 |
| ... | → 3에 점점 가까워짐 | → 0에 가까워짐 | |

- 기울기가 음수 → w를 **늘림**, 양수 → w를 **줄임**
- α가 크면 빨리 가지만 최솟값을 지나쳐 튈 수 있고, 작으면 안정적이지만 느림

**PyTorch는 미분을 자동으로 해 줌 (autograd)**

`requires_grad=True`인 텐서로 계산하면 계산 과정을 기억해 두었다가 `backward()` 한 번으로 기울기를 `.grad`에 채워 줍니다. 교안 코드에서 미분 공식을 직접 쓰지 않는 이유입니다.

In [ ]:
# f(x) = x³ → f'(x) = 3x² → x=2 에서 12
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.backward()
print(x.grad)          # tensor(12.)

# 연쇄법칙: y = (3x + 1)² → x=1 에서 24
x = torch.tensor(1.0, requires_grad=True)
y = (3 * x + 1) ** 2
y.backward()
print(x.grad)          # tensor(24.)

# −ln p 의 기울기 = −1/p → 정답 확률 0.01이면 −100 (강한 피드백)
p = torch.tensor(0.01, requires_grad=True)
(-torch.log(p)).backward()
print(p.grad)          # tensor(-100.)

In [ ]:
# 경사하강법을 직접: L(w) = (w - 3)^2
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1
for step in range(3):
    loss = (w - 3) ** 2
    loss.backward()               # 3. 역전파: w.grad = 2(w - 3)
    with torch.no_grad():
        w -= lr * w.grad          # 4. 갱신 (optimizer.step()이 하는 일)
    print(step, round(w.item(), 4))
    w.grad.zero_()                # 다음 step 전에 초기화 (optimizer.zero_grad()가 하는 일)
# 0 0.6 / 1 1.08 / 2 1.464

→ Ch5 학습 루프의 `zero_grad() → loss → backward() → step()`은 위 코드를 모든 가중치에 대해 한꺼번에 하는 것입니다.

### 0-1. 내적 · 코사인 유사도

| 개념 | 의미 | 수식 |
| --- | --- | --- |
| 내적 (Dot product) | 의미적 유사도 + 벡터 크기. 같은 방향이면 +, 반대면 −, 직교면 0 | `a · b` |
| 코사인 유사도 | 방향만의 유사도 (−1 ~ 1) | `a·b / (‖a‖‖b‖)` |
| 관계 | 단위 벡터(크기 1)에서는 내적 = 코사인 유사도 | |

Attention 점수 `Q @ K^T`는 결국 **토큰끼리의 내적 = 유사도**입니다.

In [ ]:
import torch
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([2.0, 4.0, 6.0])

dot = a @ b                                   # 28.0  (torch.dot(a, b)와 동일)
cos = dot / (a.norm() * b.norm())             # 1.0  (같은 방향)
cos2 = torch.nn.functional.cosine_similarity(a, b, dim=0)  # 1.0

### 0-2. 행렬 연산 (`*` vs `@`, transpose)

| 연산 | PyTorch | 설명 |
| --- | --- | --- |
| 요소별 곱 | `A * B` | 같은 위치끼리 곱 |
| 요소별 합 | `A + B` | 같은 위치끼리 합 (shape이 다르면 broadcast) |
| 행렬 곱 | `A @ B` | 앞 행렬의 행 · 뒤 행렬의 열 내적. `(m,n) @ (n,p) → (m,p)` |
| 전치 | `A.T`, `A.transpose(0, 1)` | 행과 열을 바꿈. `(2,3) → (3,2)` |

In [ ]:
A = torch.tensor([[1., 2.], [3., 4.]])
I = torch.tensor([[1., 0.], [0., 1.]])
A * I   # [[1,0],[0,4]]
A @ I   # [[1,2],[3,4]]

# 3D 이상에서 transpose는 바꿀 두 축을 지정
x = torch.rand(2, 6, 4)           # (b, T, d)
x.transpose(1, 2).shape           # (2, 4, 6)

**행렬 곱 shape 규칙 예시** — 안쪽 숫자가 같아야 하고, 바깥 숫자만 남습니다.

```
(4, 3) @ (3, 2) → (4, 2)        안쪽 3 = 3 ✔
(6, 2) @ (2, 6) → (6, 6)        Q @ Kᵀ 가 (T, T)가 되는 이유
(2, 6, 2) @ (2, 2, 6) → (2, 6, 6)   3D 이상이면 앞쪽 축(배치)은 그대로, 마지막 두 축만 행렬 곱
(4, 3) @ (4, 3) → 에러          안쪽 3 ≠ 4  → 한쪽을 .T 해야 함
```

### 0-2-1. `dim` 인자 · `keepdim` · shape 바꾸기

PyTorch 함수의 `dim=k`는 **"k번 축을 따라 계산해서 그 축을 없앤다(줄인다)"** 는 뜻입니다. `dim=-1`은 **마지막 축** = 벡터 하나 안에서 계산.

In [ ]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])        # (2, 3)

x.sum(dim=0)            # tensor([5., 7., 9.])   (3,)  ← 위아래(행 방향)로 더함
x.sum(dim=1)            # tensor([ 6., 15.])     (2,)  ← 좌우(한 행 안에서) 더함
x.sum(dim=-1)           # dim=1과 같음 (마지막 축)
x.sum(dim=-1, keepdim=True)   # tensor([[ 6.], [15.]])  (2, 1)  ← 축을 없애지 않고 크기 1로 남김

`keepdim=True`를 쓰는 이유: 원래 텐서와 **broadcast로 바로 연산**하기 위해서입니다.

In [ ]:
mean = x.mean(dim=-1, keepdim=True)   # (2, 1) → [[2.], [5.]]
x - mean                              # (2, 3) - (2, 1) → 각 행에서 그 행의 평균을 뺌
# tensor([[-1., 0., 1.],
#         [-1., 0., 1.]])
# keepdim 없이 (2,) 를 빼면 shape이 맞지 않아 에러 또는 엉뚱한 결과

**Broadcasting**: 크기가 1이거나 없는 축은 자동으로 늘려서 계산합니다 (뒤쪽 축부터 맞춤).

```
(8, 4, 256) + (4, 256)  → (8, 4, 256)    위치 임베딩을 배치 8개에 똑같이 더함
(2, 3)      + (2, 1)    → (2, 3)         LayerNorm의 x - mean
(2, 3)      + (3,)      → (2, 3)         모든 행에 같은 벡터를 더함 (bias)
```

**shape 바꾸기 함수**

| 함수 | 하는 일 | 예시 |
| --- | --- | --- |
| `unsqueeze(0)` | 0번 위치에 크기 1 축 추가 | `(4,)` → `(1, 4)` (배치 차원 만들기) |
| `squeeze(0)` | 0번 축이 크기 1이면 제거 | `(1, 4)` → `(4,)` |
| `view(a, b, ...)` | 원소 순서는 그대로 두고 shape만 바꿈 (원소 개수가 같아야 함) | `(2, 6)` → `(2, 3, 2)` |
| `transpose(i, j)` | i축과 j축을 맞바꿈 | `(2, 6, 4)` → `transpose(1,2)` → `(2, 4, 6)` |
| `flatten(0, 1)` | 0~1번 축을 하나로 합침 | `(2, 3, 5)` → `(6, 5)` |
| `flatten()` | 전부 1줄로 | `(2, 3)` → `(6,)` |

In [ ]:
t = torch.arange(6)          # tensor([0, 1, 2, 3, 4, 5])   (6,)
t.view(2, 3)                 # [[0, 1, 2],
                             #  [3, 4, 5]]                  (2, 3)
t.view(3, 2)                 # [[0, 1], [2, 3], [4, 5]]     (3, 2)
t.view(2, 3).T               # [[0, 3], [1, 4], [2, 5]]     (3, 2)  ← view(3,2)와 다름!
t.view(4, 2)                 # 에러: 원소 6개로 4×2=8을 만들 수 없음

> `view`는 "숫자를 다시 나눠 담기", `transpose`는 "행렬 돌리기"입니다. 결과 shape이 같아도 내용이 다를 수 있습니다.

### 0-3. 선형 함수 → 활성화 함수

#### 선형층 `nn.Linear(d_in, d_out)` 정확히 이해하기

**한 문장 정의**: 길이 `d_in`인 벡터 하나를 받아서 길이 `d_out`인 벡터 하나로 바꾸는 층입니다.

- `d_in` = 입력 벡터의 길이 = **입력 텐서의 마지막 차원 크기** (반드시 일치해야 함)
- `d_out` = 출력 벡터의 길이 = **출력 텐서의 마지막 차원 크기** (내가 정하는 값)
- 출력 벡터의 숫자 하나하나는 "입력 숫자 `d_in`개를 각기 다른 가중치로 섞은 합 + bias"입니다. 이런 '섞는 방법'이 `d_out`개 있으므로 가중치는 `d_out × d_in`개가 필요합니다.

**수식** (책마다 표기가 달라서 헷갈리는 부분)

| 표기 | 의미 |
| --- | --- |
| `z = W x + b` | x를 **세로 벡터** `(d_in,)`로 볼 때. `W`: `(d_out, d_in)` |
| `y = x Wᵀ + b` | x를 **가로(행) 벡터** / 여러 행 `(N, d_in)`로 볼 때. **PyTorch가 실제로 계산하는 식** |
| 가중치 shape | `layer.weight.shape == (d_out, d_in)` ← 순서가 **뒤집혀** 저장됨 |
| bias shape | `layer.bias.shape == (d_out,)` |
| 파라미터 수 | `d_in × d_out + d_out` (bias 포함) |

**예시 1: 숫자 3개 → 숫자 2개**

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(123)
layer = nn.Linear(3, 2)          # d_in=3, d_out=2
print(layer.weight.shape)        # torch.Size([2, 3])   ← (d_out, d_in)
print(layer.bias.shape)          # torch.Size([2])

x = torch.tensor([1.0, 2.0, 3.0])    # 길이 3 (= d_in)
y = layer(x)                         # 길이 2 (= d_out)
manual = x @ layer.weight.T + layer.bias      # (3,) @ (3, 2) → (2,)
print(torch.allclose(y, manual))     # True

직관: 가중치 행렬의 **각 행**이 "출력 숫자 하나를 만드는 레시피"입니다.

```
weight = [[w11, w12, w13],   ← y[0] = w11*x0 + w12*x1 + w13*x2 + b0
          [w21, w22, w23]]   ← y[1] = w21*x0 + w22*x1 + w23*x2 + b1
          (d_out=2 행, d_in=3 열)
```

**예시 2: 앞쪽 축은 몇 개든 상관없음 — 마지막 축만 바뀜**

`nn.Linear`는 **마지막 차원에만** 작용합니다. 앞쪽 축(배치, 토큰 수)은 그대로 통과합니다.

In [ ]:
layer = nn.Linear(3, 2)
layer(torch.rand(3)).shape         # (3,)       → (2,)
layer(torch.rand(6, 3)).shape      # (6, 3)     → (6, 2)       토큰 6개 각각 변환
layer(torch.rand(2, 6, 3)).shape   # (2, 6, 3)  → (2, 6, 2)    (b, T, d_in) → (b, T, d_out)
layer(torch.rand(2, 6, 4)).shape   # 에러! 마지막 차원 4 ≠ d_in 3

→ 그래서 Attention 코드에서 `x`가 `(b, T, d_in)`이면 `W_query(x)`는 `(b, T, d_out)`이 됩니다. 토큰 하나하나의 벡터가 독립적으로 같은 행렬과 곱해지는 것입니다.

**예시 3: 교안 코드에 나오는 모든 `nn.Linear`**

| 위치 | 코드 | d_in → d_out | 의미 | 파라미터 수 (GPT-2 Small) |
| --- | --- | --- | --- | --- |
| Ch3 실습 | `nn.Linear(3, 2)` | 3 → 2 | 3차원 임베딩을 2차원 Q/K/V로 | 6 (bias 없음) |
| MHA | `W_query = nn.Linear(d_in, d_out)` | 768 → 768 | 임베딩 → Query 벡터 | 589,824 |
| MHA | `out_proj = nn.Linear(d_out, d_out)` | 768 → 768 | head들을 합친 결과를 다시 섞음 | 590,592 |
| FFN 1층 | `nn.Linear(emb_dim, 4*emb_dim)` | 768 → 3072 | 차원을 4배로 **확장** | 2,362,368 |
| FFN 2층 | `nn.Linear(4*emb_dim, emb_dim)` | 3072 → 768 | 원래 차원으로 **축소** | 2,360,064 |
| out_head | `nn.Linear(emb_dim, vocab_size, bias=False)` | 768 → 50257 | 토큰 벡터 → 단어마다 점수(logits) | 38,597,376 |

- `d_in == d_out`이면 "같은 크기 공간 안에서 다른 표현으로 바꾸기" (Q/K/V, out_proj)
- `d_out > d_in`이면 "펼쳐서 더 세밀하게 보기" (FFN 확장)
- `d_out == vocab_size`이면 "단어마다 점수 매기기" (분류기)

> **`nn.Linear` vs `nn.Embedding` 인자 순서 비교**
> - `nn.Linear(d_in, d_out)`: weight `(d_out, d_in)` — 순서 **뒤집힘**
> - `nn.Embedding(num_embeddings, embedding_dim)`: weight `(num_embeddings, embedding_dim)` — 순서 **그대로**

#### 활성화 함수

- 활성화 함수: 비선형성을 부여해서 복잡한 패턴을 학습할 수 있게 함 (선형층만 여러 개 쌓으면 결국 선형층 하나와 같아짐)
  - Sigmoid `σ(z) = 1/(1+e^{-z})`: 층이 깊어지면 기울기 소실이 생김
  - ReLU `max(0, x)`: 층을 깊게 쌓을 수 있지만 꺼지는 뉴런이 많아짐
  - **GELU** (GPT가 사용): ReLU를 부드럽게 만든 형태라 학습이 더 안정적임 → Chapter 4 코드에서 사용

| 입력 x | ReLU | Sigmoid | GELU (대략) |
| --- | --- | --- | --- |
| -3 | 0 | 0.047 | -0.004 |
| -1 | 0 | 0.269 | -0.159 |
| 0 | 0 | 0.5 | 0 |
| 1 | 1 | 0.731 | 0.841 |
| 3 | 3 | 0.953 | 2.996 |

→ GELU는 음수를 완전히 0으로 자르지 않고 **작은 음수**를 남김 (그래서 "꺼지는 뉴런" 문제가 덜함). 양수 쪽은 ReLU와 거의 같음.

### 0-4. Softmax (+ Temperature)

- 숫자 나열을 **확률분포**로 바꿈 (모두 양수, 합 = 1). 지수함수라 작은 차이가 크게 벌어짐
- `softmax(z_i) = e^{z_i} / Σ e^{z_j}`
- 클래스가 2개인 Softmax는 Sigmoid와 같음
- **Temperature T**: `softmax(z / T)`. T가 크면 분포가 평평해짐(창의적), T가 작으면 뾰족해짐(결정적)

In [ ]:
z = torch.tensor([10., 8., 1.])
torch.softmax(z, dim=-1)          # [0.8807, 0.1192, 0.0001]
torch.softmax(z / 5.0, dim=-1)    # T=5 → [0.5448, 0.3652, 0.0901] (평평해짐)
torch.softmax(z / 0.5, dim=-1)    # T=0.5 → [0.9820, 0.0180, 0.0000] (뾰족해짐)

**손으로 한 번 계산해 보기**: `z = [1, 2, 3]`

```
e^1 = 2.718,  e^2 = 7.389,  e^3 = 20.086   합 = 30.193
softmax = [2.718/30.193, 7.389/30.193, 20.086/30.193] = [0.090, 0.245, 0.665]
```

- 입력 차이는 1씩이지만 확률은 약 2.7배씩 벌어짐 (지수함수 효과)
- `-inf`가 들어간 자리는 `e^-inf = 0` → 확률 0 (Causal mask, Top-K가 이 성질을 이용)

**`dim=-1`인 이유**: 2D 이상에서는 "어느 방향으로 합이 1이 되게 할지"를 정해야 합니다.

In [ ]:
scores = torch.tensor([[1., 2., 3.],
                       [1., 1., 1.]])
torch.softmax(scores, dim=-1)   # 각 **행**의 합 = 1 → [[0.090, 0.245, 0.665], [0.333, 0.333, 0.333]]
torch.softmax(scores, dim=0)    # 각 **열**의 합 = 1 → 의도와 다름

Attention weight `(b, T, T)`에서 행 = "지금 토큰(Query)", 열 = "참고할 토큰(Key)"이므로 **행마다 합이 1**이 되도록 `dim=-1`을 씁니다. logits `(B, V)`에서도 단어 V개에 대한 확률이므로 `dim=-1`.

### 0-5. 신경망 학습 3단계 & 경사하강법

1. **Feedforward (순전파)**: 입력 → 예측값
2. **Loss 계산**: 예측값과 실제값의 차이
3. **Backpropagation (역전파)**: Chain Rule로 각 가중치의 기울기를 계산
   - 예: 치타/사자 2배 × 사자/곰 2배 × 곰/사람 1.5배 = 치타/사람 6배
4. **가중치 갱신 (경사하강법)**: `w ← w − α · ∂L/∂w` (α = learning rate)

PyTorch에서는 이 4단계가 각각 아래 코드에 대응합니다 (Chapter 5 학습 루프).

In [ ]:
logits = model(x)                 # 1. 순전파
loss = loss_fn(logits, y)         # 2. 손실
loss.backward()                   # 3. 역전파 (gradient 계산)
optimizer.step()                  # 4. 가중치 갱신
optimizer.zero_grad()             #    다음 배치 전에 gradient 초기화 (안 하면 누적됨)

### 0-6. Cross Entropy · KL Divergence · Perplexity

- **Cross Entropy**: 정답 토큰의 확률에 로그를 취해 평균을 낸 뒤 −를 붙인 값. 정답 확률이 1에 가까울수록 0에 가까워짐
- **KL Divergence** = Cross Entropy − (데이터 자체의) Entropy → 모델이 틀린 만큼의 벌점
- **Perplexity (PPL)** = `exp(Cross Entropy)`. "매 토큰마다 평균 몇 개 후보 중에서 고민하는가"
  - 완벽한 모델: PPL = 1. K개 중 균등하게 찍는 모델: PPL = K
  - 예: 정답 확률이 0.5, 0.25, 0.125일 때 → PPL = (2×4×8)^(1/3) = 4

In [ ]:
import torch.nn.functional as F

logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3]])     # (N=2, C=3)
targets = torch.tensor([0, 1])               # 정답 클래스 인덱스

# 직접 계산
probs = torch.softmax(logits, dim=-1)
target_probs = probs[torch.arange(2), targets]   # 정답 토큰의 확률
manual = -torch.log(target_probs).mean()

# 내장 함수 (logits를 그대로 넣음! softmax 안 함)
loss = F.cross_entropy(logits, targets)
print(manual, loss)            # 같은 값
print(torch.exp(loss))         # perplexity

**위 코드를 숫자로 따라가기**

```
1번 샘플 logits [2.0, 1.0, 0.1] → softmax [0.659, 0.242, 0.099], 정답 0번 → p = 0.659 → -ln(0.659) = 0.417
2번 샘플 logits [0.5, 2.5, 0.3] → softmax [0.109, 0.803, 0.089], 정답 1번 → p = 0.803 → -ln(0.803) = 0.220
평균 loss = (0.417 + 0.220) / 2 = 0.319
perplexity = e^0.319 ≈ 1.38   (평균적으로 약 1.4개 후보 중에서 고민하는 수준 → 꽤 잘 맞춤)
```

| 정답 토큰 확률 p | loss = −ln p | 해석 |
| --- | --- | --- |
| 1.0 | 0 | 완벽 |
| 0.5 | 0.69 | 둘 중 하나 |
| 0.1 | 2.30 | |
| 1/50257 | 10.82 | GPT-2 vocab에서 무작위로 찍는 수준 (학습 전 모델) |

- `targets`는 one-hot이 아니라 **정답 클래스 번호**(정수, `torch.long`)입니다
- `F.cross_entropy`는 내부에서 `log_softmax`를 하므로 **softmax를 먼저 하면 안 됨**

### 0-7. 조건부 확률 · 베이즈 정리와 LLM

- `P(A|B) = P(A∩B) / P(B)`: 분모가 전체가 아니라 **조건을 만족하는 집합**
- 베이즈: `P(A|B) = P(B|A)P(A) / P(B)`. Prior에 Evidence를 더해 Posterior를 얻음
- LLM 관점:
  - 다음 토큰 예측 = `P(w_t | w_<t)`
  - 프롬프트는 새로운 증거 → 답변 분포가 바뀜
  - RAG는 외부 증거를 문맥에 넣어 환각을 줄임

## 1. 벡터 공간 (Chapter 1)

### 이론

- **Embedding**: 데이터(단어)를 벡터로 표현한 것
- 비슷한 의미의 단어는 벡터 공간에서 가까이 위치 → 단어 의미를 **계산**할 수 있음
  - `King − Man + Woman ≈ Queen`
- 유사도는 코사인 유사도로 측정

### 핵심 코드 (`gensim`)

In [ ]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-50")        # 사전학습 단어 벡터 (50차원)

model.similarity('cat', 'dog')                    # 두 단어의 코사인 유사도
model.most_similar('computer', topn=5)            # 가장 유사한 단어 5개 → [(word, score), ...]
model.doesnt_match(['apple', 'banana', 'orange', 'car'])  # 'car' (그룹에 안 어울리는 단어)

### 간단 실습

In [ ]:
# 단어 산술: king - man + woman = ?
model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)  # [('queen', ...)]

# 벡터 직접 보기
vec = model['cat']       # numpy array, shape (50,)

| 메서드 | 반환 |
| --- | --- |
| `similarity(w1, w2)` | float (코사인 유사도) |
| `most_similar(w, topn=k)` | `[(단어, 점수), ...]` 리스트 |
| `doesnt_match(list)` | 그룹에서 벗어난 단어 하나 (str) |

## 2. 데이터셋: 토크나이저 · 슬라이딩 윈도우 · 임베딩 (Chapter 2)

### 이론

**전처리 흐름**: 텍스트 → Tokenizer(토큰 ID) → Token Embedding + Positional Embedding → 모델 입력

**Tokenizer 종류**

| 종류 | 방식 |
| --- | --- |
| BPE (GPT 계열) | 가장 많이 등장하는 쌍을 찾아 어휘에 추가 |
| WordPiece (BERT) | 각각 나올 확률보다 쌍으로 나올 확률이 높으면 추가 |
| Unigram | 최선의 조합을 찾고 쓸모없는 것은 버림. 확률적 샘플링이라 결과가 매번 다를 수 있고 속도가 느림 |

- 자주 쓰이는 단어는 길게 자름 (압축률 향상). 가능하면 의미 단위로 자름
- 한국어는 EXAONE, 중국어는 Qwen 토크나이저가 효율적 (같은 문장을 더 적은 토큰으로 표현)

**Special Token**: 모델을 제어하고 입출력 구조를 정의하는 신호 (예: `<|endoftext|>`, bos/eos/pad). 학습 대상에 포함되지만 Padding Token은 제외. 모델마다 다름

**위치 인코딩**: 토큰 자체에는 위치 정보가 없음 → `Tom ate the meat ≠ The meat ate Tom`

| 방식 | 특징 |
| --- | --- |
| 절대 위치 (학습형) | GPT-2. `nn.Embedding(context_length, emb_dim)`을 더함 → **교안 코드 방식** |
| Sinusoidal | 삼각함수 사용. 낮은 차원은 빠르게, 높은 차원은 천천히 변함 |
| 상대 위치 | 토큰 사이의 거리를 bias로 더함. 캐싱이 어려워 느림 |
| RoPE | 2개씩 짝지어 각도를 다르게 **회전**시킴 (최신 LLM) |

**데이터셋 = Self-Supervised**: 라벨이 따로 없고 텍스트 자체에서 (입력, 정답)을 만듦. 정답은 입력을 **한 칸 오른쪽으로 민 것** (다음 토큰 예측).

```
입력  x: [ 40,  367, 2885, 1464]
정답  y: [367, 2885, 1464, 1807]   ← x를 한 칸 shift
```

### 핵심 코드 ① tiktoken 인코딩/디코딩

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})   # str → list[int]
strings  = tokenizer.decode(integers)                                  # list[int] → str

- `allowed_special`을 지정하지 않으면 `<|endoftext|>`가 포함된 텍스트에서 에러가 남
- `<|endoftext|>`의 ID = 50256 (GPT-2 vocab_size = 50257)

### 핵심 코드 ② x / y 만들기

In [ ]:
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

### 핵심 코드 ③ `GPTDatasetV1` (★ 통째로 외우기)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

포인트:
- `range(0, len(token_ids) - max_length, stride)`: 마지막 target이 범위를 넘지 않도록 `- max_length`
- `stride == max_length`이면 겹침 없음, `stride < max_length`이면 겹침 (데이터 중복)
- `Dataset`을 상속하면 `__len__`, `__getitem__` 두 개를 반드시 구현

**인자 의미**

| 인자 | 의미 |
| --- | --- |
| `txt` | 원본 텍스트 전체 (문자열 하나) |
| `max_length` | 샘플 하나(입력 x)의 토큰 수 = 모델이 한 번에 보는 길이 |
| `stride` | 다음 샘플을 만들 때 시작 위치를 몇 칸 옮길지 |

**슬라이딩 윈도우 예시**: `token_ids = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]` (10개), `max_length = 4`

```
stride=4  → range(0, 6, 4) = [0, 4]          샘플 2개, 겹침 없음
  i=0: x=[10 11 12 13]  y=[11 12 13 14]
  i=4: x=[14 15 16 17]  y=[15 16 17 18]
  (19는 x에 들어가지 못하고 버려짐)

stride=2  → range(0, 6, 2) = [0, 2, 4]       샘플 3개, 2칸씩 겹침
  i=0: x=[10 11 12 13]  y=[11 12 13 14]
  i=2: x=[12 13 14 15]  y=[13 14 15 16]
  i=4: x=[14 15 16 17]  y=[15 16 17 18]

stride=1  → range(0, 6, 1) = [0..5]          샘플 6개, 3칸씩 겹침 (데이터 최대한 활용, 과적합 위험↑)
```

- 샘플 하나 안에서 예측 문제가 `max_length`개 들어 있음: `10→11`, `10 11→12`, `10 11 12→13`, `10 11 12 13→14`
- `len(token_ids) - max_length`까지만 도는 이유: `i=6`이면 y가 `[17 18 19 ?]`로 범위를 넘음

### 핵심 코드 ④ `create_dataloader_v1` (★)

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

- `drop_last=True`: 마지막 배치가 batch_size보다 작으면 버림 (loss가 튀는 것 방지)

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
# [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

### 핵심 코드 ⑤ Token Embedding + Positional Embedding

In [ ]:
# 작은 예제
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size, output_dim = 6, 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)   # weight shape (6, 3)
token_embeddings = embedding_layer(input_ids)                  # (4, 3) = weight에서 2,3,5,1번 행 조회

# 실제 크기
vocab_size, output_dim = 50257, 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
inputs, targets = next(iter(dataloader))          # inputs: (8, 4)
token_embeddings = token_embedding_layer(inputs)  # (8, 4, 256)

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))     # (4, 256)

input_embeddings = token_embeddings + pos_embeddings               # (8, 4, 256)  ← broadcast

- `nn.Embedding(num_embeddings, embedding_dim)`: 첫 인자는 **개수**(vocab 또는 위치 수), 둘째 인자는 **차원**
- Embedding = "ID로 행을 꺼내는 lookup table" (one-hot @ weight와 같음)
- 위치 임베딩의 입력은 `torch.arange(seq_len)` = `[0, 1, 2, 3]`

**`nn.Embedding(6, 3)`을 표로 보기** — 행 6개(ID 0~5) × 열 3개(벡터 길이)

```
weight (6, 3)                 input_ids = [2, 3, 5, 1]
 ID 0: [ a0  a1  a2 ]
 ID 1: [ b0  b1  b2 ]  ─┐     출력 (4, 3)
 ID 2: [ c0  c1  c2 ]  ─┼──→  [ c0 c1 c2 ]   ← ID 2
 ID 3: [ d0  d1  d2 ]  ─┤     [ d0 d1 d2 ]   ← ID 3
 ID 4: [ e0  e1  e2 ]   │     [ f0 f1 f2 ]   ← ID 5
 ID 5: [ f0  f1  f2 ]  ─┘     [ b0 b1 b2 ]   ← ID 1
```

- 입력 shape 뒤에 `embedding_dim`이 붙는다고 기억: `(4,)` → `(4, 3)`, `(8, 4)` → `(8, 4, 256)`
- ID가 `num_embeddings` 이상이면 `IndexError` (예: `nn.Embedding(6, 3)`에 ID 6을 넣으면 에러)
- 그래서 위치 임베딩은 `nn.Embedding(context_length, emb_dim)` → 위치 0 ~ context_length-1까지만 가능 → **입력이 context_length보다 길면 안 되는 이유**

| 임베딩 | `num_embeddings` (행 수) | `embedding_dim` (열 수) | 입력 |
| --- | --- | --- | --- |
| 토큰 임베딩 | `vocab_size` = 50257 (토큰 종류 수) | `emb_dim` = 256 / 768 | 토큰 ID `(B, T)` |
| 위치 임베딩 | `context_length` = 4 / 1024 (위치 개수) | `emb_dim` (토큰 임베딩과 **같아야** 더할 수 있음) | `torch.arange(T)` `(T,)` |

**토큰 + 위치를 더할 때의 broadcast**

```
token_embeddings (8, 4, 256)    문장 8개 × 토큰 4개 × 256
pos_embeddings      (4, 256)    위치 4개 × 256  → 앞에 배치 축이 없으므로 8개 문장 모두에 똑같이 더해짐
결과            (8, 4, 256)
```

즉 "모든 문장의 첫 번째 토큰에는 위치0 벡터, 두 번째 토큰에는 위치1 벡터..."를 더합니다. 같은 단어 "the"라도 문장 맨 앞이냐 세 번째냐에 따라 최종 벡터가 달라집니다.

### Hugging Face 모델 파일 구조

| 파일 | 비유 | 역할 |
| --- | --- | --- |
| `config.json` | 설계도 | 레이어 수, hidden size, head 수 등 모델 구조 |
| `generation_config.json` | 말하기 습관 | max_length, temperature, top_p, eos_token_id |
| `model.safetensors` | 뇌 | 학습된 가중치 (`.bin`보다 빠르고 안전) |
| `vocab.json` / `merges.txt` | 단어장 / BPE 병합 규칙 | |
| `tokenizer.json` | 번역기 | vocab + merges 통합본 (빠름) |
| `tokenizer_config.json` | | 토크나이저 클래스, 특수 토큰 정의 |

In [ ]:
from transformers import AutoTokenizer
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
ids = qwen.encode("안녕하세요, 반갑습니다!", add_special_tokens=True)
pieces = [qwen.decode([t]) for t in ids]
qwen.eos_token, qwen.eos_token_id      # 특수 토큰 확인 (bos_token, unk_token, pad_token 등)

## 3. Attention (Chapter 3)

### 이론

- **Attention** = 문맥 파악. 서로 관련된 토큰일수록 점수가 커짐
  - 예: "애플"이 과일인지 회사인지는 주변 토큰을 보고 결정
- 임베딩에 가중치를 곱해 Q, K, V를 만듦

| | 역할 | 비유 |
| --- | --- | --- |
| **Query** | 질문을 던지는 주체 | "내가 무엇을 찾는가" |
| **Key** | 질문에 답하는 대상 | "나는 어떤 단서인가" (어디를 볼지 정하는 검색용) |
| **Value** | 전달할 정보 | "무엇을 가져올지" (실제 내용) |

- **Q와 K를 분리하는 이유**: 같은 벡터를 쓰면 내적이 대칭이라(`Tom·meat = meat·Tom`) 누가 누구를 참고하는지 구분하지 못하고, 항상 자기 자신을 가장 크게 주목함
- **K와 V를 분리하는 이유**: 검색 단서와 전달할 내용이 섞이지 않게 하기 위해

**Scaled Dot-Product Attention**

```
Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) V
```

- **Scale (`/ √d_k`)**: 차원 d_k가 커지면 내적값의 표준편차가 √d_k만큼 커짐 → 나눠서 분산을 1로 맞춤 (softmax가 한쪽으로 쏠리지 않게 함)
- **Causal Masking**: 현재 토큰이 **미래 토큰을 보지 못하도록** 대각선 위쪽을 `-inf`로 채움 → softmax 후 0
- **Dropout**: softmax 이후 attention weight 일부를 무작위로 0으로 만듦 (과적합 방지)
- **Context vector**: attention weight로 Value를 가중합한 것 → `z(2) = α21·v(1) + α22·v(2) + ...`
- **Multi-head**: 여러 head가 각각 다른 관점에서 문맥을 파악
  - GPT-2 Small: 12 head × 12 Transformer block
  - 초기 Layer는 문맥 파악, 후반 Layer는 다음 토큰 예측에 주력

### 숫자로 보는 Attention (가중치 없는 단순 버전)

교안 예제 문장 "Your journey starts with one step" (토큰 6개, 각 3차원)에서 **"journey"(2번째 토큰)가 Query**일 때:

In [ ]:
query = inputs[1]                       # journey = [0.55, 0.87, 0.66]
attn_scores_2 = inputs @ query          # (6, 3) @ (3,) → (6,)  모든 토큰과 내적
# [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]
#  Your    journey starts  with    one     step

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
# [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]   합 = 1

context_vec_2 = attn_weights_2 @ inputs # (6,) @ (6, 3) → (3,)
# [0.4419, 0.6515, 0.5683]

- 1단계 **점수**: 내적 = 얼마나 비슷한가 → journey 자신(1.495)과 비슷한 starts(1.475)가 높음
- 2단계 **가중치**: softmax로 합이 1인 비율로 바꿈
- 3단계 **context vector**: 모든 토큰 벡터를 비율대로 섞은 것 = "문맥이 반영된 새로운 journey 벡터"
- 실제 코드에서는 `inputs`를 그대로 쓰지 않고 `W_query`, `W_key`, `W_value`(학습되는 Linear)를 거친 Q, K, V를 쓰고, `√d_k`로 나누고, mask를 씌운다는 점만 다릅니다

**모든 토큰을 한 번에 (행렬 버전)**

In [ ]:
attn_scores = inputs @ inputs.T          # (6, 3) @ (3, 6) → (6, 6)
attn_weights = torch.softmax(attn_scores, dim=-1)   # 각 행의 합 = 1
all_context_vecs = attn_weights @ inputs # (6, 6) @ (6, 3) → (6, 3)

`(6, 6)` 행렬의 `[i, j]` = "i번째 토큰이 j번째 토큰을 얼마나 참고하는가".

### CausalAttention의 `d_in`, `d_out`

| 인자 | 의미 | 교안 예제 | GPT-2 Small |
| --- | --- | --- | --- |
| `d_in` | 들어오는 토큰 벡터의 길이 (= 임베딩 차원) | 3 | 768 |
| `d_out` | Q, K, V 벡터 하나의 길이 = 출력 context vector 길이 | 2 | 768 |
| `context_length` | mask 행렬을 만들 최대 크기 `(context_length, context_length)` | 6 | 1024 |
| `dropout` | attention weight를 0으로 만들 비율 | 0.0 | 0.1 |
| `qkv_bias` | Q/K/V Linear에 bias를 쓸지 | False | False |

- 교안 예제는 3 → 2로 줄여서 계산 과정을 눈으로 보기 쉽게 한 것이고, 실제 GPT에서는 **residual 연결 때문에 `d_in == d_out`** 입니다
- `context_length`는 "최대", 실제 입력 길이는 `forward`에서 `num_tokens`로 얻음 → 그래서 mask를 `[:num_tokens, :num_tokens]`로 자름

### Tensor shape 흐름 (CausalAttention)

교안 예제 값 (b=2, T=6, d_in=3, d_out=2)을 괄호 안에 함께 적었습니다.

```
x            : (b, T, d_in)                      (2, 6, 3)
Q, K, V      : (b, T, d_out)            ← nn.Linear(d_in, d_out)        (2, 6, 2)
K.transpose(1,2) : (b, d_out, T)                                         (2, 2, 6)
scores = Q @ Kᵀ  : (b, T, T)            ← 토큰 × 토큰 유사도             (2, 6, 6)
mask[:T, :T]     : (T, T)  상삼각(대각선 위) = 1                          (6, 6)
weights = softmax(scores/√d_out, dim=-1) : (b, T, T)   각 행의 합 = 1    (2, 6, 6)
context = weights @ V : (b, T, d_out)                                    (2, 6, 2)
```

- `transpose(1, 2)`인 이유: K는 `(b, T, d_out)`이고 배치 축(0)은 그대로 두고 **마지막 두 축만** 바꿔야 `(T, d_out) @ (d_out, T)`가 됨. `.T`를 쓰면 배치 축까지 뒤집혀서 틀림
- 출력의 마지막 차원이 `d_out`인 것은 V가 `d_out` 길이 벡터이기 때문

### 핵심 코드 ① `CausalAttention` (★ 통째로 외우기)

In [ ]:
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

포인트:
- `register_buffer`: 학습 대상(parameter)은 아니지만 `state_dict`에 저장되고 `.to(device)` 때 함께 이동함
- `torch.triu(..., diagonal=1)`: 대각선은 0, **대각선 위만 1** (= 미래 위치)
- `masked_fill_` (밑줄 = in-place). 마스크는 `.bool()`로 바꿔서 넣음
- `[:num_tokens, :num_tokens]`: 입력이 context_length보다 짧을 때 크기를 맞춤
- 순서: **mask → scale + softmax → dropout → @ values**

### 간단 실습: 마스크 눈으로 확인

In [ ]:
T = 4
mask = torch.triu(torch.ones(T, T), diagonal=1)
# tensor([[0., 1., 1., 1.],
#         [0., 0., 1., 1.],
#         [0., 0., 0., 1.],
#         [0., 0., 0., 0.]])

scores = torch.rand(T, T)
scores.masked_fill_(mask.bool(), -torch.inf)
w = torch.softmax(scores, dim=-1)
print(w)            # 대각선 위 = 0, 각 행 합 = 1
print(w.sum(-1))    # tensor([1., 1., 1., 1.])

**행 하나를 따라가 보기** (2번째 토큰, 즉 행 index 1)

```
mask 행      [0,   0,   1,    1   ]   → 1인 자리(미래)를 가림
scores 행    [0.9, 0.3, 0.7,  0.5 ]
masked       [0.9, 0.3, -inf, -inf]
softmax      [0.646, 0.354, 0, 0]    ← 자기 자신과 이전 토큰만 참고, 합 = 1
```

- 왜 0이 아니라 `-inf`로 채우나? 0으로 채우면 `e^0 = 1`이라 확률이 생김. `-inf`여야 `e^-inf = 0`
- 왜 softmax **전에** mask하나? softmax 후에 0으로 만들면 남은 값들의 합이 1이 아니게 됨
- 첫 번째 토큰(행 0)은 자기 자신만 볼 수 있어서 weight가 `[1, 0, 0, 0]`

### 실행 예제

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]] # step
)
d_in = inputs.shape[1]   # 3
d_out = 2
batch = torch.stack((inputs, inputs), dim=0)    # (2, 6, 3)

torch.manual_seed(123)
context_length = batch.shape[1]                 # 6
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)                        # (2, 6, 2)

### 핵심 코드 ② `MultiHeadAttentionWrapper` (단순 버전)

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

- head를 여러 개 만들어 마지막 차원으로 이어 붙임 → 출력 `(b, T, d_out * num_heads)`
- 위 예제(`d_out=2, num_heads=2`)의 결과: `(2, 6, 4)`
- `nn.ModuleList`를 써야 내부 파라미터가 모델에 등록됨 (파이썬 list를 쓰면 등록 안 됨)

### 핵심 코드 ③ `MultiHeadAttention` (★★ 가장 중요. Chapter 4에서도 그대로 사용)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

**MHA shape 흐름 (외워 둘 것)**

```
x                         (b, T, d_in)
W_key(x)                  (b, T, d_out)
.view(b, T, H, hd)        (b, T, H, hd)        d_out = H * hd
.transpose(1, 2)          (b, H, T, hd)        ← head를 배치처럼 앞으로
Q @ K.transpose(2, 3)     (b, H, T, T)         ← 2D가 아니라 4D라서 (2, 3)
weights @ V               (b, H, T, hd)
.transpose(1, 2)          (b, T, H, hd)        ← 원래 순서로
.contiguous().view(b, T, d_out)                ← head 합치기
out_proj                  (b, T, d_out)
```

**작은 숫자로 view → transpose 따라가기** (b=1, T=3, d_out=4, num_heads=2 → head_dim=2)

```
W_query(x) 결과 (1, 3, 4):
  토큰0: [ 1,  2,  3,  4]
  토큰1: [ 5,  6,  7,  8]
  토큰2: [ 9, 10, 11, 12]

.view(1, 3, 2, 2)  → 각 토큰 벡터 4칸을 [head0 2칸 | head1 2칸]으로 나눔
  토큰0: head0 [1, 2]   head1 [3, 4]
  토큰1: head0 [5, 6]   head1 [7, 8]
  토큰2: head0 [9,10]   head1 [11,12]

.transpose(1, 2)  → (1, 2, 3, 2)  head 단위로 다시 묶음
  head0: 토큰0 [1, 2], 토큰1 [5, 6], 토큰2 [ 9,10]   ← head0은 이 (3, 2) 행렬로 attention 수행
  head1: 토큰0 [3, 4], 토큰1 [7, 8], 토큰2 [11,12]   ← head1은 이 (3, 2) 행렬로 attention 수행
```

- head마다 `(T, head_dim) @ (head_dim, T)` = `(T, T)` attention 행렬이 따로 생김 → `(b, H, T, T)`
- 끝에서는 반대로 `transpose(1, 2)` → `view(b, T, d_out)`으로 다시 `[head0 | head1]`을 이어 붙임
- GPT-2 Small: `d_out=768`, `num_heads=12` → `head_dim=64`. 768칸을 64칸씩 12조각으로 나눠 각 head가 담당
- `d_out % num_heads != 0`이면 (예: 768을 10으로) 나눠떨어지지 않아 assert 에러

Wrapper와 비교:

| | `MultiHeadAttentionWrapper` | `MultiHeadAttention` |
| --- | --- | --- |
| 방식 | head마다 CausalAttention 객체를 따로 만들어 concat | Linear 하나로 만든 뒤 `view`로 head를 쪼갬 (병렬, 효율적) |
| 출력 차원 | `d_out * num_heads` | `d_out` |
| out_proj | 없음 | 있음 |
| 전치 | `transpose(1, 2)` | `transpose(2, 3)` (4D) |

자주 헷갈리는 것:
- `head_dim = d_out // num_heads` (`/`가 아니라 `//`)
- `transpose` 다음에 `view`를 하려면 메모리가 연속적이어야 해서 `.contiguous()`가 필요함
- `keys.shape[-1]`은 MHA에서 `head_dim`임 (√head_dim으로 scale)

### 참고: Attention 시각화 (forward hook)

In [ ]:
hooks = [block.att.register_forward_hook(make_hook()) for block in model.trf_blocks]
with torch.no_grad():
    model(input_tensor)
for hook in hooks:
    hook.remove()
# hook_fn(module, input, output): input[0]이 att 모듈의 입력 x

## 4. GPT 모델 조립 (Chapter 4)

### 이론

GPT는 **Transformer의 Decoder만** 사용 (Causal LM: 앞쪽 문맥만 보고 다음 토큰을 맞춤).
BERT는 Encoder만 사용 (Masked LM: 빈칸 채우기, 양방향 문맥).

```
in_idx → tok_emb + pos_emb → dropout
       → [TransformerBlock] × n_layers
       → final_norm → out_head(Linear emb_dim → vocab_size) → logits
```

**TransformerBlock 구조 (Pre-LayerNorm)**

```
x ─┬─ LayerNorm → MultiHeadAttention → Dropout ─(+)─┬─ LayerNorm → FeedForward → Dropout ─(+)→ out
   └──────────────── shortcut ───────────────────┘   └──────────── shortcut ──────────────┘
```

| 구성 요소 | 역할 |
| --- | --- |
| **LayerNorm** | 각 토큰 벡터를 평균 0, 분산 1로 맞춰 학습 안정화. 마지막에 학습 파라미터 γ(scale), β(shift) 적용. ε은 0으로 나누기 방지 |
| RMSNorm (참고) | 평균 빼기를 생략하고 크기(scale)만 맞춤. 최신 LLM이 사용 |
| **GELU** | 부드러운 ReLU. `0.5x(1 + tanh(√(2/π)(x + 0.044715x³)))` |
| **FeedForward** | "지식의 저장소". emb_dim → **4배** 확장 → GELU → 다시 emb_dim. 차원을 확장해 특징을 세분화 |
| SwiGLU (참고) | Swish 게이트 × value. 최신 LLM FFN에서 사용 |
| **Residual (shortcut)** | 입력을 출력에 더함 → 변화량만 학습하면 되고 기울기 소실을 막음. 그래서 **입력 shape = 출력 shape** 이어야 함 |
| **Dropout** | 과적합 방지 |

**GPT 입출력**

- 모든 Transformer 블록의 입출력 차원이 같음 (블록을 쌓고 residual을 쓰기 위해)
- 마지막 `out_head`에서만 `emb_dim → vocab_size`로 바뀜
- 예: B=1, T=4, d=768, V=50257 → logits `(1, 4, 50257)`
- **마지막 위치의 logits에서 가장 큰 값의 인덱스 = 다음 토큰 ID**

**GPT-2 Small (124M) 설정**

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # 단어 집합 크기
    "context_length": 1024,  # 최대 문맥 길이
    "emb_dim": 768,          # 임베딩 차원
    "n_heads": 12,           # 어텐션 헤드 수
    "n_layers": 12,          # 레이어 수
    "drop_rate": 0.1,        # 드롭아웃 비율
    "qkv_bias": False        # Q,K,V 편향 사용 여부
}

| 키 | 정확한 의미 | 어디에 쓰이나 |
| --- | --- | --- |
| `vocab_size` | 토큰 ID 종류 수. tiktoken GPT-2 토크나이저와 일치해야 함 | `tok_emb` 행 수, `out_head`의 d_out |
| `context_length` | 한 번에 넣을 수 있는 최대 토큰 수 | `pos_emb` 행 수, mask 크기, `generate`의 자르기 길이 |
| `emb_dim` | 모든 층에서 토큰 하나를 표현하는 벡터 길이 | 거의 모든 층의 d_in/d_out |
| `n_heads` | head 개수. `emb_dim`을 나눠떨어지게 해야 함 (768/12=64) | MHA의 `num_heads` |
| `n_layers` | TransformerBlock 개수 | `trf_blocks` |
| `drop_rate` | Dropout 비율 (10%를 0으로) | 임베딩 뒤, attention weight, shortcut 앞 |
| `qkv_bias` | Q/K/V Linear의 bias 여부 | MHA의 `qkv_bias` |

참고로 같은 구조에서 크기만 키운 GPT-2 계열:

| 모델 | emb_dim | n_layers | n_heads |
| --- | --- | --- | --- |
| Small (124M) | 768 | 12 | 12 |
| Medium (355M) | 1024 | 24 | 16 |
| Large (774M) | 1280 | 36 | 20 |
| XL (1558M) | 1600 | 48 | 25 |

### 핵심 코드 ① LayerNorm · GELU · FeedForward (★)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))    # γ
        self.shift = nn.Parameter(torch.zeros(emb_dim))   # β

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

포인트:
- LayerNorm: `dim=-1` (마지막 차원 = 임베딩 차원 기준), `keepdim=True` (broadcast용), `unbiased=False` (N으로 나눔)
- `scale`은 ones, `shift`는 zeros로 초기화. `nn.Parameter`로 감싸야 학습됨
- GELU 상수: `2.0 / torch.pi`, `0.044715`, `x³ = torch.pow(x, 3)`

**LayerNorm 숫자 예시**: 토큰 벡터 `x = [1, 2, 3, 4]`

```
mean = (1+2+3+4)/4 = 2.5
var  = ((-1.5)² + (-0.5)² + 0.5² + 1.5²) / 4 = 1.25        ← unbiased=False라 N=4로 나눔
norm_x = (x - 2.5) / √1.25 = [-1.342, -0.447, 0.447, 1.342]  ← 평균 0, 분산 1
출력 = scale * norm_x + shift   (처음엔 scale=1, shift=0 이라 norm_x 그대로)
```

- 토큰마다 **따로** 정규화합니다 (`dim=-1`). 배치 전체나 다른 토큰의 값은 섞이지 않음
- `(B, T, emb_dim)` 입력이면 `mean`은 `(B, T, 1)` → keepdim 덕분에 broadcast로 빼기 가능

**FeedForward shape 예시** (`emb_dim=768`)

```
x (2, 4, 768) → Linear(768, 3072) → (2, 4, 3072) → GELU → (2, 4, 3072) → Linear(3072, 768) → (2, 4, 768)
```

입력과 출력 shape이 같아서 `x + shortcut`이 가능합니다.

### 간단 실습: LayerNorm 검증

In [ ]:
torch.manual_seed(123)
x = torch.randn(2, 5)
ln = LayerNorm(emb_dim=5)
out = ln(x)
print(out.mean(dim=-1))                      # ≈ [0, 0]
print(out.var(dim=-1, unbiased=False))       # ≈ [1, 1]

# 내장 nn.LayerNorm과 비교
print(torch.allclose(out, nn.LayerNorm(5)(x), atol=1e-5))   # True

### 핵심 코드 ② TransformerBlock (★)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

- 패턴은 두 번 반복: `shortcut = x → norm → (att | ff) → dropout → x + shortcut`
- MHA에서 `d_in = d_out = emb_dim` (입출력 shape이 같아야 residual 가능)
- config 키 이름 주의: `n_heads`(MHA 인자명은 `num_heads`), `drop_rate`(인자명 `dropout`)

### 핵심 코드 ③ GPTModel (★★)

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

포인트:
- `nn.Sequential(*[...])`: 리스트 앞의 `*`로 풀어서 넣음
- `torch.arange(seq_len, device=in_idx.device)`: 위치 텐서를 입력과 같은 device에 생성
- `out_head`는 `bias=False`
- shape: `in_idx (B, T)` → `tok_embeds (B, T, 768)` + `pos_embeds (T, 768)` → ... → `logits (B, T, 50257)`

### 핵심 코드 ④ `generate_text_simple` (★ Greedy decoding)

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

```
idx          (B, T)
idx_cond     (B, ≤context_size)     ← 뒤에서부터 context_size만큼만 자름
logits       (B, T, V)
logits[:, -1, :]  (B, V)            ← 마지막 위치 = 다음 토큰 예측
idx_next     (B, 1)                 ← keepdim=True라 2차원 유지 → cat 가능
idx          (B, T+1)
```

- softmax는 단조 증가 함수라 argmax 결과가 같으므로 생략해도 됨
- **Autoregressive**: 생성한 토큰을 다시 입력에 붙여서 반복

### 실행 예제

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()                                   # dropout 끔

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode("Hello, I am")       # [15496, 11, 314, 716]
encoded_tensor = torch.tensor(encoded).unsqueeze(0)   # (1, 4)  배치 차원 추가

out = generate_text_simple(model=model, idx=encoded_tensor,
                           max_new_tokens=10,
                           context_size=GPT_CONFIG_124M["context_length"])
decoded_text = tokenizer.decode(out.squeeze(0).tolist())   # 학습 전이라 엉뚱한 문장 출력

### 간단 실습: 파라미터 수 · shape 확인

In [ ]:
batch = torch.tensor([[6109, 3626, 6100, 345],
                      [6109, 1110, 6622, 257]])      # (2, 4)
print(model(batch).shape)                            # torch.Size([2, 4, 50257])

total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,}")   # 163,009,536 (out_head와 tok_emb의 가중치를 공유하면 약 124M)

**163,009,536은 어디서 나오나** (Linear 파라미터 = `d_in × d_out + d_out`)

| 부분 | 계산 | 파라미터 수 |
| --- | --- | --- |
| `tok_emb` | 50257 × 768 | 38,597,376 |
| `pos_emb` | 1024 × 768 | 786,432 |
| 블록 1개 – W_query/W_key/W_value | 3 × (768 × 768) (bias 없음) | 1,769,472 |
| 블록 1개 – out_proj | 768 × 768 + 768 | 590,592 |
| 블록 1개 – FFN | (768×3072 + 3072) + (3072×768 + 768) | 4,722,432 |
| 블록 1개 – LayerNorm 2개 | 2 × (768 + 768) | 3,072 |
| **블록 1개 합** | | 7,085,568 |
| 블록 × 12 | | 85,026,816 |
| `final_norm` | 768 + 768 | 1,536 |
| `out_head` | 768 × 50257 (bias 없음) | 38,597,376 |
| **총합** | | **163,009,536** |

- 163,009,536 − 38,597,376(out_head) = 124,412,160 ≈ **124M** (가중치 공유 시)
- 파라미터의 절반 가까이가 블록 안의 FFN에 있음 → "FFN = 지식의 저장소"

## 5. 사전학습 Pretraining (Chapter 5)

### 이론

**학습 과정**: Dataset 준비 → 순전파 → Loss 계산 → 역전파 → 가중치 갱신 → (주기적으로) 검증

| 개념 | 설명 |
| --- | --- |
| Train / Validation 분리 | Validation은 과적합 감지와 하이퍼파라미터 결정에 사용. 교안에서는 텍스트를 **9:1**로 분리 |
| 비용함수 | 수치 예측은 MSE, 분류(다음 토큰 예측)는 **Cross Entropy** |
| Cross Entropy 계산 | 정답 토큰들의 로그 확률 → 평균 → − 부호 |
| 경사하강법 | Loss를 최소화하는 가중치를 찾는 과정. Optimizer가 스스로 조율 |
| Local minimum | 고차원에서는 거의 발생하지 않음. 오히려 너무 평탄해서 방향을 못 찾는 기울기 소실이 문제 |
| **Adam** | Momentum(관성: 방향) + RMSprop(파라미터별 보폭 조절). **AdamW** = Adam + Weight Decay(규제) |
| Epoch | 전체 데이터를 1회 학습 |
| Step (Iteration) | 파라미터를 1회 업데이트 |
| Batch size | Step 1회에 쓰는 데이터 개수. 1,000개 / batch 100 → 1 epoch = 10 step |
| 작은 batch | 불안정하지만 일반화에 유리, 메모리 적음, 시간 증가 |
| 큰 batch | 안정적이지만 과적합 위험, 메모리 많음, 시간 단축 |
| Training vs Inference | Training은 (Masking 덕분에) **모든 위치를 한 번에** 계산, Inference는 Autoregressive로 한 토큰씩 |

**디코딩 전략**

| 방식 | 설명 |
| --- | --- |
| Greedy | 매번 `argmax` → 결정적, 반복적인 문장이 나오기 쉬움 |
| Temperature | `logits / T` 후 softmax → `torch.multinomial`로 샘플링. T↑ 다양, T↓ 결정적 |
| Top-K | 상위 K개만 남기고 나머지는 `-inf` → 그 안에서 확률적으로 선택 |

### Loss 계산 시 shape (★ 가장 많이 묻는 부분)

```
logits         (B, T, V)    →  logits.flatten(0, 1)   (B*T, V)
target_batch   (B, T)       →  target_batch.flatten() (B*T,)
F.cross_entropy(입력 (N, C), 정답 (N,))
```

`cross_entropy`는 `(N, C)` 형태의 입력과 `(N,)` 형태의 정답 인덱스를 요구하므로, 배치와 시퀀스 차원을 합쳐서 평탄화합니다.

**작은 예시** (B=2 문장, T=3 토큰, V=5 단어)

```
logits (2, 3, 5)                          target_batch (2, 3)
 문장0: 토큰0 [5개 점수]                    [[1, 4, 2],
        토큰1 [5개 점수]                     [0, 3, 3]]
        토큰2 [5개 점수]
 문장1: 토큰0~2 ...

logits.flatten(0, 1) → (6, 5)   "예측 문제 6개, 각각 5개 중 하나 고르기"
target_batch.flatten() → (6,)   [1, 4, 2, 0, 3, 3]   각 문제의 정답 번호
```

- `flatten(0, 1)`: 0번(B)과 1번(T) 축만 합치고 마지막 V 축은 유지
- `flatten()`: 인자가 없으면 전부 1줄로
- 결과 loss는 6개 예측 문제의 평균 (스칼라 하나)

### 핵심 코드 ① `calc_loss_batch` (★)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    return loss

### 핵심 코드 ② `calc_loss_loader` · `evaluate_model` (★)

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

- `loss.item()`: 텐서에서 파이썬 float만 꺼냄 (그래프와 연결이 끊겨 메모리 절약)
- 평가 패턴: **`model.eval()` → `torch.no_grad()` → ... → `model.train()`**
  - `eval()`: Dropout 등을 끔
  - `no_grad()`: gradient 계산을 끔 (메모리 절약, 속도 향상)

### 핵심 코드 ③ `train_model_simple` (★★ 학습 루프)

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen = 0
    global_step = -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

**학습 4줄 (순서까지 암기)**

In [ ]:
optimizer.zero_grad()                                           # 1. 이전 gradient 초기화
loss = calc_loss_batch(input_batch, target_batch, model, device)  # 2. 순전파 + loss
loss.backward()                                                 # 3. 역전파
optimizer.step()                                                # 4. 가중치 갱신

- `input_batch.numel()`: 원소 개수 = B × T = 이번 배치에서 본 토큰 수
- `global_step = -1`에서 시작 → 첫 step이 0이 되어 `0 % eval_freq == 0`으로 바로 평가

**인자 의미**

| 인자 | 의미 | 교안 값 |
| --- | --- | --- |
| `num_epochs` | 전체 train 데이터를 몇 바퀴 돌지 | 10 |
| `eval_freq` | **몇 step마다** train/val loss를 측정할지 | 5 |
| `eval_iter` | 측정할 때 **배치 몇 개**로 loss를 계산할지 (전부 계산하면 느리니까) | 1 |
| `start_context` | 매 epoch 끝에 이어 쓰기 샘플을 만들 시작 문장 | `"Every effort moves you"` |

예: batch_size=2, context_length=256이면 1 step에 `2 × 256 = 512` 토큰을 봄 → `tokens_seen`이 512씩 증가. step 0, 5, 10, 15 ...에서 평가 출력.

### 핵심 코드 ④ 텍스트 ↔ 토큰 변환 · 샘플 출력

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)   # (T,) → (1, T)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)                           # (1, T) → (T,)
    return tokenizer.decode(flat.tolist())


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]          # = context_length
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        print(decoded_text.replace("\n", " "))
    model.train()

- `unsqueeze(0)`는 배치 차원 추가, `squeeze(0)`은 배치 차원 제거
- `model.pos_emb.weight.shape[0]`: 위치 임베딩의 행 개수 = 모델이 처리할 수 있는 최대 길이

### 핵심 코드 ⑤ `generate` (교안 노트북 버전: Greedy + EOS)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

`generate_text_simple`과 다른 점은 **EOS 토큰이 나오면 중단**하는 것 하나뿐입니다.

**참고: Temperature + Top-K 버전** (`previous_chapters.py`의 `generate`, 슬라이드 이론에 해당)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]                    # k번째로 큰 값
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            logits = logits - logits.max(dim=-1, keepdim=True).values   # 수치 안정화
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)          # 확률적 샘플링
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

### 핵심 코드 ⑥ `main`: 데이터 분리 · Optimizer · DataLoader

In [ ]:
def main(gpt_config, settings):
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. 데이터 (text_data = the-verdict.txt 내용)
    ...

    # 2. 모델 + 옵티마이저
    model = GPTModel(gpt_config)
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"]
    )

    # 3. 9:1 분리 후 DataLoader
    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))

    train_loader = create_dataloader_v1(
        text_data[:split_idx],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=True,
        shuffle=True,
        num_workers=0
    )
    val_loader = create_dataloader_v1(
        text_data[split_idx:],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=False,
        shuffle=False,
        num_workers=0
    )

    # 4. 학습
    tokenizer = tiktoken.get_encoding("gpt2")
    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=settings["num_epochs"], eval_freq=5, eval_iter=1,
        start_context="Every effort moves you", tokenizer=tokenizer
    )
    return train_losses, val_losses, tokens_seen, model

| | train_loader | val_loader |
| --- | --- | --- |
| 데이터 | `text_data[:split_idx]` | `text_data[split_idx:]` |
| `shuffle` | **True** (매 epoch 섞음) | **False** (평가는 일관되게) |
| `drop_last` | **True** | **False** (검증 데이터를 버리지 않음) |

- `max_length = stride = context_length` → 겹치지 않는 청크
- 하이퍼파라미터: `lr=5e-4`, `num_epochs=10`, `batch_size=2`, `weight_decay=0.1`, `context_length=256` (속도를 위해 1024에서 줄임)

### 핵심 코드 ⑦ 모델 저장 / 로드

In [ ]:
torch.save(model.state_dict(), "outputs/model.pth")                 # 저장

model = GPTModel(GPT_CONFIG_124M)                                     # 1. 빈 모델 생성 (CPU)
model.load_state_dict(torch.load("outputs/model.pth", weights_only=True))  # 2. 가중치 로드
model.to(device)                                                      # 3. device로 이동

- 저장하는 것은 모델 객체가 아니라 `state_dict()` (파라미터 딕셔너리)
- 로드할 때는 **같은 config로 구조를 먼저 만들고** 가중치를 넣음

### 간단 실습: Loss와 Perplexity 감 잡기

In [ ]:
# 학습 전 모델의 loss ≈ ln(50257) ≈ 10.8 (50257개 중 무작위로 찍는 수준)
import math
print(math.log(50257))          # 10.82

torch.manual_seed(123)
model = GPTModel({**GPT_CONFIG_124M, "context_length": 256}).to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
print(train_loss, math.exp(train_loss))   # loss ≈ 11, PPL ≈ 수만 → 학습이 진행되면 감소

### 간단 실습: Temperature / Top-K 직접 해 보기

In [ ]:
next_token_logits = torch.tensor([4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79])

# Greedy
torch.argmax(next_token_logits).item()            # 3

# Temperature 샘플링
torch.manual_seed(123)
for T in [0.1, 1.0, 5.0]:
    probs = torch.softmax(next_token_logits / T, dim=0)
    samples = [torch.multinomial(probs, num_samples=1).item() for _ in range(1000)]
    print(T, torch.bincount(torch.tensor(samples), minlength=9))
# T=0.1 → 거의 전부 3, T=5 → 여러 토큰이 고르게 나옴

# Top-K (k=3)
top_logits, top_pos = torch.topk(next_token_logits, 3)   # 값 [6.75, 6.28, 4.51], 위치 [3, 7, 0]
new_logits = torch.where(next_token_logits < top_logits[-1],
                         torch.tensor(float("-inf")), next_token_logits)
torch.softmax(new_logits, dim=0)                          # 상위 3개 외에는 확률 0

## 6. 분류 미세 튜닝 · LoRA (Chapter 6)

### 이론

**Fine-tuning(미세 튜닝)** = 사전학습으로 언어를 이미 아는 모델을 가져와서, 특정 작업에 맞게 **조금 더** 학습시키는 것.

| 종류 | 목표 | 데이터 | 교안 실습 |
| --- | --- | --- | --- |
| **분류 미세 튜닝** | 정해진 클래스 중 하나 고르기 (스팸/정상) | 문장 + 라벨(0/1) | Chapter 6 |
| **지시 미세 튜닝** (SFT) | 지시를 따라 자유롭게 답하기 | 지시 + 정답 응답 | Chapter 7 |
| **선호 튜닝** (DPO) | 더 좋은 답을 고르도록 | 지시 + 좋은 답 + 나쁜 답 | Chapter 7 DPO |

**분류 미세 튜닝의 4가지 포인트** (슬라이드 177~181)

1. **마지막 layer 교체**: `out_head`를 `Linear(768 → 50257)`에서 `Linear(768 → 2)`로 바꿈. "다음 단어 50257개 중 무엇?"이 "클래스 2개 중 무엇?"으로 바뀜
2. **Padding**: 문장 길이가 다 다르므로 짧은 문장 뒤를 `<|endoftext|>`(50256)로 채워 길이를 맞춤 (배치로 묶으려면 길이가 같아야 함)
3. **Target은 0/1 라벨**: `ham → 0`, `spam → 1`. 다음 토큰 ID가 아니라 **클래스 번호**
4. **결과는 마지막 토큰의 출력 → softmax → argmax** (softmax는 argmax 결과를 바꾸지 않으므로 코드에서는 생략)

**왜 마지막 토큰의 출력만 쓰나?**

Causal mask 때문에 각 토큰은 **자기 자신과 앞쪽 토큰만** 볼 수 있습니다. 문장 전체를 본 토큰은 **마지막 토큰 하나뿐**이므로, 문장 전체에 대한 판단은 마지막 위치의 출력으로 합니다.

```
"You won a free prize"  →  토큰 5개
  토큰0 "You"   : "You"만 봄
  토큰2 "a"     : "You won a"까지 봄
  토큰4 "prize" : 문장 전체를 봄  ← 이 위치의 출력 (2개 점수)로 스팸 여부 판단
```

**파라미터 동결(Freeze)**: 모든 파라미터를 다 학습하면 느리고 과적합 위험이 큼 → 대부분은 `requires_grad = False`로 고정하고 **뒤쪽 일부만** 학습.

| 부분 | requires_grad | 이유 |
| --- | --- | --- |
| 임베딩, 블록 0~10 | False (고정) | 사전학습으로 이미 언어를 잘 이해함 |
| 마지막 블록 `trf_blocks[-1]` | **True** | 작업에 맞게 표현을 조금 조정 |
| `final_norm` | **True** | 마지막 블록과 함께 조정 |
| 새 `out_head` (768→2) | **True** (새로 만든 층은 기본값이 True) | 처음부터 학습해야 하는 분류기 |

> 순서 주의: **① 전체 freeze → ② out_head 교체 → ③ 마지막 블록과 final_norm unfreeze**. out_head를 freeze보다 먼저 교체하면 새 out_head까지 얼어 버립니다.

### 데이터 흐름 (SMS Spam Collection)

```
원본 약 5,570개 (ham 약 4,830 / spam 747)  → 불균형
  ↓ create_balanced_dataset : ham을 spam 수(747)만큼만 무작위 추출 (언더샘플링)
균형 1,494개 (ham 747 / spam 747)
  ↓ Label map {"ham": 0, "spam": 1}
  ↓ random_split(df, 0.7, 0.1)
train 1,045 (70%) / validation 149 (10%) / test 300 (나머지 20%)
  ↓ SpamDataset : 토큰화 → 자르기 → 패딩
  ↓ DataLoader(batch_size=8)
input (8, max_length), label (8,)
```

- 불균형 데이터에서는 "전부 ham"이라고만 답해도 정확도 87%가 나오므로 균형을 맞춤
- val/test의 `max_length`는 **train의 max_length를 그대로 사용** (모델에 들어가는 길이를 일관되게)

### 핵심 코드 ① 데이터 준비 함수

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

- `df.sample(frac=1, ...)`: 전체(100%)를 무작위 순서로 뽑기 = **섞기**
- `reset_index(drop=True)`: 섞인 뒤 인덱스를 0부터 다시 매김
- 1,494개 기준: `train_end = int(1494*0.7) = 1045`, `validation_end = 1045 + int(1494*0.1) = 1045 + 149 = 1194`

### 핵심 코드 ② `SpamDataset` (★ 빈칸 출제)

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

빈칸 정답 위치:

| 빈칸 | 정답 |
| --- | --- |
| 자르기 | `encoded_text[:self.max_length]` |
| 패딩 | `[pad_token_id] * (self.max_length - len(encoded_text))` |
| 라벨 컬럼 | `self.data.iloc[index]["Label"]` |

**패딩 예시** (`max_length = 6`, `pad_token_id = 50256`)

```
"Hi there"          → [17250, 612]                     길이 2 → 50256을 4개 붙임
                    → [17250, 612, 50256, 50256, 50256, 50256]
"Free entry now!!"  → [11146, 5726, 783, 3228]         길이 4 → 50256을 2개 붙임
아주 긴 문장 (10토큰) → 앞 6개만 남기고 자름 (max_length가 주어졌을 때만)
```

- `[x] * 3` = `[x, x, x]` (리스트 반복). `* 0`이면 빈 리스트라 이미 길이가 맞는 문장은 그대로
- `__getitem__`은 **(입력 토큰 텐서, 라벨 텐서)** 를 반환. 라벨은 스칼라 텐서 `tensor(1)`
- `dtype=torch.long`: `nn.Embedding`의 입력과 `cross_entropy`의 정답은 정수(long)여야 함

### 핵심 코드 ③ `calc_accuracy_loader` · `calc_loss_batch` (★)

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

**Chapter 5와 Chapter 6의 `calc_loss_batch` 비교** (★ 자주 헷갈림)

| | Chapter 5 (사전학습) | Chapter 6 (분류) |
| --- | --- | --- |
| 모델 출력 | `(B, T, 50257)` | `(B, T, 2)` |
| 사용하는 위치 | **모든** 위치 | **마지막** 위치만 `[:, -1, :]` → `(B, 2)` |
| 정답 | `(B, T)` 다음 토큰 ID | `(B,)` 클래스 번호 0/1 |
| loss 호출 | `cross_entropy(logits.flatten(0, 1), target.flatten())` | `cross_entropy(logits, target_batch)` (이미 (N, C), (N,) 형태라 flatten 불필요) |

**정확도 계산 예시** (batch 4개)

```
logits[:, -1, :] = [[ 2.1, -0.5],    → argmax 0 (ham)
                    [-1.0,  3.2],    → argmax 1 (spam)
                    [ 0.3,  0.9],    → argmax 1
                    [ 1.5,  1.4]]    → argmax 0
predicted = [0, 1, 1, 0],  target = [0, 1, 0, 0]
(predicted == target) = [True, True, False, True] → .sum() = 3 → 정확도 3/4 = 75%
```

> 노트북 실행 주의: 위 셀을 실행하면 Chapter 5의 `calc_loss_batch`가 분류용 버전으로 **덮어써집니다**. Chapter 5 예제를 다시 돌리려면 Chapter 5의 정의 셀을 다시 실행하세요.

In [ ]:
# 위 예시를 직접 실행
logits = torch.tensor([[2.1, -0.5], [-1.0, 3.2], [0.3, 0.9], [1.5, 1.4]])
target = torch.tensor([0, 1, 0, 0])
pred = torch.argmax(logits, dim=-1)                 # tensor([0, 1, 1, 0])
print((pred == target).sum().item() / len(target))  # 0.75
print(torch.nn.functional.cross_entropy(logits, target))

### 핵심 코드 ④ `train_classifier_simple` (★)

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

Chapter 5의 `train_model_simple`과 다른 점:

| | `train_model_simple` (Ch5) | `train_classifier_simple` (Ch6) |
| --- | --- | --- |
| 세는 단위 | `tokens_seen += input_batch.numel()` (토큰 수 B×T) | `examples_seen += input_batch.shape[0]` (**문장 수** B) |
| epoch 끝 | `generate_and_print_sample` (문장 생성) | `calc_accuracy_loader` (정확도 출력) |
| 반환 | losses, tokens_seen | losses, **accs**, examples_seen |
| 학습 4줄 | 동일 | 동일 |

### 핵심 코드 ⑤ 모델 수정 (★ 빈칸 출제)

In [ ]:
# 1. 전체 파라미터 동결
for param in model.parameters():
    param.requires_grad = False

torch.manual_seed(123)

# 2. 출력층 교체: 768 → 50257  ⇒  768 → 2
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

model.to(device)

# 3. 마지막 Transformer 블록과 final_norm만 학습 허용
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

# 4. 학습
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

- `in_features`, `out_features` = `nn.Linear(d_in, d_out)`의 정식 인자 이름. `in_features=768`이 `d_in`, `out_features=2`가 `d_out`
- `trf_blocks[-1]`: `nn.Sequential`도 리스트처럼 인덱싱 가능. `-1` = 마지막 블록
- optimizer에 `model.parameters()` 전체를 넘겨도 `requires_grad=False`인 파라미터는 gradient가 없어서 갱신되지 않음
- 교안 설정: GPT-2 Small(124M), `qkv_bias=True` (OpenAI 가중치에 bias가 있어서), `lr=5e-5`, `batch_size=8`, `num_epochs=5`

**학습되는 파라미터 수 확인**

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{trainable:,} / {total:,}")

| 학습되는 부분 | 계산 | 파라미터 수 |
| --- | --- | --- |
| 마지막 블록 (qkv_bias=True) | 7,085,568 + 3×768 | 7,087,872 |
| final_norm | 768 + 768 | 1,536 |
| 새 out_head | 768×2 + 2 | 1,538 |
| **합** | | **7,090,946** (전체 약 124M의 약 5.7%) |

### 이론: LoRA (Low-Rank Adaptation)

**문제**: LLM 전체 학습은 비싸다. 10B 모델 학습에 약 160GB 메모리 필요 (가중치 + gradient + Adam 상태).

**아이디어**: 원래 가중치 `W`는 **고정**하고, 변화량 `ΔW`만 작은 행렬 두 개의 곱 `A @ B`로 학습.

```
원래:   y = x W
LoRA:   y = x W + (alpha / rank) · x A B
             └고정┘   └────── 학습 ──────┘

W : (in_dim, out_dim)         예: (768, 768)  = 589,824개  (고정)
A : (in_dim, rank)            예: (768, 16)   =  12,288개  (학습)
B : (rank, out_dim)           예: (16, 768)   =  12,288개  (학습)
A @ B : (in_dim, out_dim)     예: (768, 768)  ← W와 같은 크기의 변화량을 만들어 냄
```

→ 학습 파라미터 589,824개 대신 24,576개 (**약 4.2%**)

| 용어 | 의미 |
| --- | --- |
| **rank (r)** | A와 B 사이의 "좁은 통로" 크기. 클수록 표현력↑, 파라미터↑ (교안 16) |
| **alpha (α)** | LoRA 출력의 세기 조절. 실제 배율은 `alpha / rank` (교안 16/16 = 1.0) |
| Low-rank | `A @ B`는 768×768이지만 rank가 최대 16인 행렬 → 적은 숫자로 큰 행렬의 변화를 근사 |

**초기화 규칙** (슬라이드 188, ★ 이유까지)

| A | B | 결과 |
| --- | --- | --- |
| 랜덤 | **0** | ✔ 교안 방식. 처음엔 `A @ B = 0` → 사전학습 모델과 **완전히 같은 출력**에서 출발 |
| 랜덤 | 랜덤 | ✘ 처음부터 W에 노이즈가 더해짐 → 사전학습 효과가 망가짐 |
| 0 | 0 | ✘ 두 행렬 모두 기울기가 0 → 영원히 학습 안 됨 |

- 첫 역전파에서는 **B만** 업데이트됨 (B=0이라 A의 기울기가 0). 이후 A, B 모두 학습

**QLoRA**: 고정된 원래 가중치 W를 **4-bit로 양자화**해서 메모리를 크게 줄이고, 계산할 때만 BF16으로 복원(dequantize)해서 LoRA를 학습. 원래는 GPU에 올릴 수 없던 큰 모델도 학습 가능.

**BF16**: 지수부는 FP32와 같고(표현 범위 동일) 가수부 bit가 적음(정밀도↓). 학습 중 값의 범위가 정규화 덕분에 일정해서 정밀도가 조금 낮아도 충분.

### 핵심 코드 ⑥ `LoRALayer` · `LinearWithLoRA` · `replace_linear_with_lora` (★★ 빈칸 출제)

In [ ]:
import math
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

빈칸 정답 위치:

| 빈칸 | 정답 | 외우는 법 |
| --- | --- | --- |
| A shape | `torch.empty(in_dim, rank)` | 입력 쪽 → 좁은 통로 |
| A 초기화 | `nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))` | 랜덤 (nn.Linear 기본 초기화와 같은 방식) |
| B shape | `torch.zeros(rank, out_dim)` | 좁은 통로 → 출력 쪽, **zeros** |
| forward | `(self.alpha / self.rank) * (x @ self.A @ self.B)` | A 먼저, B 나중 |
| 합치기 | `self.linear(x) + self.lora(x)` | 원래 출력 + 변화량 (residual과 같은 모양) |
| 교체 대상 | `isinstance(module, torch.nn.Linear)` | |
| 교체 | `setattr(model, name, LinearWithLoRA(module, rank, alpha))` | `model.name = ...`과 같음 |

포인트:
- `nn.Parameter`로 감싸야 학습 대상으로 등록됨 (Ch4 LayerNorm의 scale/shift와 같음)
- `linear.in_features`, `linear.out_features`: 기존 Linear의 d_in, d_out을 그대로 읽어 와서 A, B 크기를 맞춤
- `named_children()`: **바로 아래** 자식 모듈만 (이름, 모듈) 쌍으로 돌려줌 → Linear가 아니면 그 안으로 **재귀** 호출해서 깊숙한 Linear까지 찾음
- 주의: `nn.Linear`의 weight는 `(out, in)`으로 저장되지만 LoRA의 A는 `(in_dim, rank)`라서 `x @ A`를 바로 계산할 수 있음 (전치 불필요)

**shape 따라가기** (GPT-2 Small의 `W_query`, rank=16)

```
x            (B, T, 768)
x @ A        (B, T, 768) @ (768, 16) → (B, T, 16)     좁은 통로로 압축
  @ B        (B, T, 16)  @ (16, 768) → (B, T, 768)    다시 원래 크기로
× alpha/rank (B, T, 768)
linear(x)    (B, T, 768)                              고정된 원래 출력
합           (B, T, 768)                              shape 변화 없음 → 모델 구조에 그대로 끼워 넣을 수 있음
```

In [ ]:
# LoRA가 처음엔 출력에 영향이 없는지 확인 (B = 0)
torch.manual_seed(123)
linear = nn.Linear(768, 768)
lora_linear = LinearWithLoRA(linear, rank=16, alpha=16)
x = torch.randn(2, 4, 768)
print(torch.allclose(linear(x), lora_linear(x)))           # True
print(lora_linear.lora.A.shape, lora_linear.lora.B.shape)  # (768, 16), (16, 768)

### 핵심 코드 ⑦ LoRA 적용 순서 (`main`의 핵심부)

In [ ]:
model = GPTModel(BASE_CONFIG)
model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
model.to(device)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)

for param in model.parameters():
    param.requires_grad = False

replace_linear_with_lora(model, rank=LORA_RANK, alpha=LORA_ALPHA)
model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 학습 가능 LoRA 파라미터 수: {total_params:,}")    # 2,666,528

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)

순서: **① out_head 교체 → ② 전체 freeze → ③ LoRA 주입** (LoRA의 A, B는 freeze 이후에 새로 만들어지므로 `requires_grad=True`)

> 일반 분류 튜닝(⑤)과 순서가 다릅니다. 여기서는 out_head를 먼저 바꾸고 freeze하므로 **새 out_head도 고정**되고, out_head 역시 Linear이므로 LoRA가 붙어서 LoRA로만 학습됩니다.

**2,666,528은 어디서 나오나** (LoRA 파라미터 = `rank × (in_dim + out_dim)`)

| Linear | 개수 | in → out | 개당 16×(in+out) | 합 |
| --- | --- | --- | --- | --- |
| W_query, W_key, W_value, out_proj | 4 × 12블록 | 768 → 768 | 24,576 | 1,179,648 |
| FFN 확장 | 1 × 12 | 768 → 3072 | 61,440 | 737,280 |
| FFN 축소 | 1 × 12 | 3072 → 768 | 61,440 | 737,280 |
| out_head | 1 | 768 → 2 | 12,320 | 12,320 |
| **합** | | | | **2,666,528** |

| | 일반 분류 튜닝 | LoRA |
| --- | --- | --- |
| 학습 대상 | 마지막 블록 + final_norm + out_head | 모든 Linear에 붙은 A, B |
| 학습 파라미터 | 약 709만 | 약 267만 |
| learning rate | 5e-5 | 8e-4 (파라미터가 적어서 더 크게) |
| 교안 결과 | | 테스트 정확도 약 97.7% |

## 7. 지시 미세 튜닝 · DPO (Chapter 7)

### 이론: 지시 미세 튜닝 (Instruction Fine-tuning, SFT)

사전학습 모델은 "다음 단어 잇기"만 잘함 → 질문을 줘도 질문을 이어 쓰거나 엉뚱한 문장을 생성. **(지시, 정답 응답) 쌍**으로 학습시켜서 지시를 따르는 Chat 모델로 만듦.

**데이터 형식** (`instruction-data.json`, 1,100개)

```json
{
  "instruction": "Evaluate the following phrase by transforming it into the spelling given.",
  "input": "freind --> friend",
  "output": "The spelling of the given phrase \"freind\" is incorrect, the correct spelling is \"friend\"."
}
```

- `input`은 비어 있을 수 있음 (`""`) → 그때는 `### Input:` 부분을 아예 넣지 않음
- 분할: train 85% (935) / test 10% (110) / validation 나머지 (55)

**Prompt 형식 (Alpaca 스타일)** — 학습과 추론에서 **같은 형식**을 써야 함

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".
```

- `format_input(entry)` = 위에서 `### Response:` **앞까지**
- `InstructionDataset`은 응답까지 붙인 **전체 텍스트**를 토큰화 → Chapter 5와 똑같이 "다음 토큰 예측"으로 학습

**학습 원리는 Chapter 5와 같음**: 입력과 정답을 한 칸 shift → `cross_entropy`. 다른 점은 **배치 만들기(collate)** 뿐.

**Padding과 Masking** (슬라이드 200~203)

| 단계 | 하는 일 | 이유 |
| --- | --- | --- |
| Padding(Input) | 배치 안 가장 긴 문장 길이에 맞춰 50256으로 채움 | 텐서로 쌓으려면 길이가 같아야 함 |
| Padding(Target) | input을 한 칸 shift해서 target 생성 | 다음 토큰 예측 |
| Masking(Target) | target의 패딩 중 **첫 번째 50256만 남기고** 나머지는 `-100` | 첫 50256은 "여기서 답변 끝"을 학습시키기 위해 남김. 나머지 패딩은 loss에서 제외 |
| Masking(Instruction) (선택) | 지시 부분 target도 `-100` | 지시문을 외우지 말고 응답만 학습. 교안 코드에는 구현되지 않음 (DPO에서 비슷한 마스킹 사용) |

**`-100`의 의미**: `F.cross_entropy`의 기본값 `ignore_index=-100` → 정답이 -100인 위치는 **loss 계산에서 빠짐**.

### 핵심 코드 ① `format_input` · `InstructionDataset` (★)

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- `SpamDataset`과 달리 여기서는 **패딩하지 않음**. `__getitem__`은 길이가 제각각인 **파이썬 리스트**를 반환 → 패딩은 collate 함수에서 배치마다 함
- 이유: 전체 최대 길이에 맞추면 짧은 배치도 쓸데없이 길어짐. **배치 안에서 가장 긴 것**에만 맞추면 효율적
- 줄바꿈 `\n\n### Instruction:\n`, `\n\n### Input:\n`, `\n\n### Response:\n` 형식 그대로 외우기

In [ ]:
# format_input 실행 결과 확인
entry = {"instruction": "Convert to passive voice.", "input": "The cat chased the mouse.",
         "output": "The mouse was chased by the cat."}
print(format_input(entry) + f"\n\n### Response:\n{entry['output']}")

entry_no_input = {"instruction": "Name a primary color.", "input": "", "output": "Red."}
print(format_input(entry_no_input))   # ### Input: 부분이 없음

### 핵심 코드 ② `custom_collate_fn` (★★ 빈칸 출제)

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

**한 줄씩 예시로 따라가기**: `batch = [[0, 1, 2, 3, 4], [5, 6], [7, 8, 9]]`, `P = 50256`

```
batch_max_length = max(5+1, 2+1, 3+1) = 6        ← +1은 끝에 붙일 50256 자리

item [5, 6]
  new_item += [P]            → [5, 6, P]                       (길이 3)
  padded                     → [5, 6, P, P, P, P]              (길이 6)
  inputs  = padded[:-1]      → [5, 6, P, P, P]                 (길이 5)
  targets = padded[1:]       → [6, P, P, P, P]                 (길이 5)
  mask = targets == P        → [F, T, T, T, T]
  indices = nonzero(mask)    → [1, 2, 3, 4]
  targets[indices[1:]] = -100  → 첫 P(위치 1)는 남기고 2, 3, 4를 -100
  targets                    → [6, P, -100, -100, -100]
```

배치 전체 결과:

```
inputs  = [[0, 1, 2, 3, 4],          targets = [[1, 2, 3, 4,    P   ],
           [5, 6, P, P, P],                     [6, P, -100, -100, -100],
           [7, 8, 9, P, P]]                     [8, 9, P,    -100, -100]]
```

In [ ]:
# 직접 실행해서 확인
inputs, targets = custom_collate_fn([[0, 1, 2, 3, 4], [5, 6], [7, 8, 9]])
print(inputs)
print(targets)

빈칸 정답 위치:

| 빈칸 | 정답 |
| --- | --- |
| 끝 토큰 추가 | `new_item += [pad_token_id]` |
| 패딩 | `[pad_token_id] * (batch_max_length - len(new_item))` |
| 입력 | `padded[:-1]` |
| 정답 | `padded[1:]` |
| 마스킹 | `targets[indices[1:]] = ignore_index` (첫 번째 패딩은 남김) |

포인트:
- `item.copy()`: 원본 리스트를 바꾸지 않기 위해 복사 (Dataset에 저장된 데이터가 매 epoch 길어지는 버그 방지)
- `torch.nonzero(mask)`: True인 위치 인덱스를 `(개수, 1)` 형태로 반환 → `.squeeze()`로 1차원
- `indices.numel() > 1`: 패딩이 2개 이상일 때만 마스킹 (1개면 그건 "끝" 표시라 남김). 패딩이 1개면 squeeze 결과가 0차원 텐서라 `[1:]` 슬라이싱도 불가능
- `torch.stack`: 같은 길이의 1D 텐서들을 쌓아 `(B, L)` 2D 텐서로
- `allowed_max_length=1024`: 모델의 context_length를 넘지 않도록 자름

**`partial`로 인자 고정 후 DataLoader에 전달**

In [ ]:
from functools import partial

customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

- DataLoader는 collate 함수를 `collate_fn(batch)` **인자 하나로만** 호출 → `device` 같은 추가 인자는 `partial`로 미리 채워 둠
- `partial(f, device=device)` = "device가 이미 채워진 새 함수"

**`ignore_index=-100` 확인**

In [ ]:
import torch.nn.functional as F
logits = torch.tensor([[2.0, 1.0], [0.5, 1.5], [1.0, 1.0]])   # 3개 위치, 클래스 2개
t1 = torch.tensor([0, 1, 1])
t2 = torch.tensor([0, 1, -100])                               # 3번째 위치 무시

print(F.cross_entropy(logits, t1))         # 3개 평균
print(F.cross_entropy(logits, t2))         # 앞 2개만의 평균
print(F.cross_entropy(logits[:2], t1[:2])) # 위와 같은 값

### 핵심 코드 ③ 지시 튜닝 `main`의 학습 · 응답 생성 부분

In [ ]:
# 모델: GPT-2 Medium (355M)
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model = load_gpt2_model("gpt2-medium-355M.pth", BASE_CONFIG)
model.to(device)

# 학습: Chapter 5의 train_model_simple을 그대로 사용
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs = 2
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

# 테스트 데이터에 대한 응답 생성
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
    test_data[i]["model_response"] = response_text

포인트:
- 모델 구조, loss 함수, 학습 루프 **모두 Chapter 4~5 그대로**. 바뀐 것은 데이터(Dataset + collate)뿐 → "SFT = 형식을 갖춘 데이터로 하는 추가 사전학습"
- `eos_id=50256`: 모델이 `<|endoftext|>`를 생성하면 멈춤 → collate에서 첫 번째 50256을 마스킹하지 않고 남긴 이유
- `generated_text[len(input_text):]`: 생성 결과에는 **프롬프트가 앞에 포함**되어 있으므로 프롬프트 길이만큼 잘라 내고 응답만 추출
- `.replace("### Response:", "").strip()`: 응답 머리말과 앞뒤 공백 제거
- `start_context=format_input(val_data[0])`: epoch마다 검증 데이터 첫 지시에 대한 응답을 출력해 학습 진행을 눈으로 확인

### 이론: 선호 튜닝 (RLHF / PPO / DPO)

SFT로 "지시를 따르는" 모델은 만들었지만, 답이 무례하거나 성의 없을 수 있음 → **사람이 더 좋아하는 답**을 하도록 추가 학습.

```
Prompt : 여름 휴가지를 추천해줘
Chosen  (👍 선호): 제주도의 함덕 해변은 어떠신가요? (상세 설명...)
Rejected(👎 비선호): 바다로 가세요. (성의 없음)
```

| 방식 | 과정 | 특징 |
| --- | --- | --- |
| **RLHF (PPO)** | ① 사람 평가로 **Reward Model**(AI 채점관) 학습 → ② 모델이 답을 생성하면 Reward Model이 점수 → ③ 강화학습(PPO)으로 점수가 높아지게 | 모델이 여러 개 필요, 복잡, 불안정 |
| **DPO** (Direct Preference Optimization) | Reward Model 없이 **(prompt, chosen, rejected) 데이터만으로 직접** 학습 | 단순, 일반 학습 루프와 비슷 |

**DPO의 핵심 아이디어**

1. **Bradley-Terry 모델**: "A가 B보다 선호될 확률" = `sigmoid(점수A − 점수B)`. 점수 차이가 클수록 확률이 1에 가까워짐
2. 점수 = 정책 모델(policy)이 그 답을 생성할 **로그 확률**이 기준 모델(reference)보다 얼마나 높아졌는가
3. **KL 제약 역할**: reference 모델과 비교하므로 모델이 원래 모습에서 **너무 급격히 벗어나지 않음** (말을 이상하게 하면서 점수만 올리는 것을 방지)
4. **로그를 쓰는 이유**: 문장 확률 = 토큰 확률들의 **곱** → 로그를 취하면 **덧셈**이 되어 계산이 안정적

**DPO Loss**

```
logits = [log π(chosen) − log π(rejected)] − [log π_ref(chosen) − log π_ref(rejected)]
           └────── policy가 chosen을 더 좋아하는 정도 ──────┘   └──── reference의 정도 ────┘
loss = −log sigmoid(β · logits)
```

- `logits > 0`: policy가 reference보다 chosen 쪽으로 더 기울었음 → loss 감소
- **β (beta)**: 얼마나 강하게 밀어붙일지 (작을수록 reference 근처에 머무름). 교안 0.1
- 학습 시작 시 policy = reference(같은 가중치)라서 `logits = 0` → `loss = −log sigmoid(0) = −log 0.5 = ln 2 ≈ 0.693` (교안 출력 첫 줄의 `Train loss 0.694`)

| 모델 | 역할 | 학습 | 코드 |
| --- | --- | --- | --- |
| `policy_model` | 학습시킬 모델 | ✔ | gradient 계산 O |
| `reference_model` | 비교 기준 (보통 SFT 완료 모델. 교안 코드는 `gpt2-medium-355M.pth`를 로드). policy와 같은 가중치로 시작 | ✘ 고정 | `.eval()` + `torch.no_grad()` |

### 핵심 코드 ④ `PreferenceDataset` (★ 빈칸 출제)

데이터 형식 (`instruction-data-with-preference.json`): 기존 필드 + `chosen`, `rejected`

```json
{
  "instruction": "Evaluate the following phrase by transforming it into the spelling given.",
  "input": "freind --> friend",
  "output": "...",
  "rejected": "The spelling of the given phrase \"freind\" is flat out wrong, get it together, the correct spelling is \"friend\".",
  "chosen": "The spelling of the given phrase \"freind\" is incorrect, the correct spelling is \"friend\"."
}
```

In [ ]:
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []

        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{chosen_response}")
            rejected_full_tokens = tokenizer.encode(f"{prompt}\n\n### Response:\n{rejected_response}")

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- 샘플 하나 = **딕셔너리** `{"prompt", "chosen", "rejected"}` (모두 토큰 ID 리스트)
- `prompt_tokens`를 따로 저장하는 이유: 나중에 **프롬프트 길이만큼 마스킹**하기 위해

### 핵심 코드 ⑤ DPO용 `custom_collate_fn`

In [ ]:
def custom_collate_fn(batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu"):
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []
    }

    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key]) + 1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))

            mask = torch.ones(len(padded)).bool()
            mask[len(sequence):] = False

            if mask_prompt_tokens:
                mask[:prompt.shape[0] + 2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = torch.stack(batch_data[key])
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        batch_data[key] = tensor_stack.to(device)

    return batch_data

**SFT collate와 비교**

| | SFT `custom_collate_fn` | DPO `custom_collate_fn` |
| --- | --- | --- |
| 반환 | `(inputs, targets)` 튜플 | 딕셔너리 (`chosen`, `rejected`, `chosen_mask`, `rejected_mask`, `prompt`) |
| 무시할 위치 표시 | target에 `-100` | 별도의 **bool mask** (False = 무시) |
| shift | collate에서 `[:-1]`, `[1:]` | collate에서 안 함 → `compute_logprobs`에서 함 |
| 프롬프트 마스킹 | 안 함 | `mask[:len(prompt) + 2] = False` (응답만 평가) |

**mask 예시**: prompt 토큰 4개, chosen 전체 8개, `max_length_common = 10`

```
위치      0  1  2  3  4  5  6  7  8  9
chosen   p  p  p  p  h  h  r  r  P  P      p=prompt, h=머리말 앞 2토큰("\n\n", "###"), r=나머지(Response:\n + 응답), P=패딩
기본     T  T  T  T  T  T  T  T  F  F      mask[len(sequence):] = False   (패딩 제외)
프롬프트  F  F  F  F  F  F  T  T  F  F      mask[:4 + 2] = False            (프롬프트 + 2토큰 제외)
```

- `+2`는 프롬프트 바로 뒤에 붙는 `\n\n###` 2토큰까지 함께 제외하기 위함 (교안 주석: "+2는 \n\n때문")
- chosen과 rejected를 **같은 길이**(`max_length_common`)로 맞춤

> 노트북 실행 주의: 교안과 똑같이 SFT와 **같은 이름**(`custom_collate_fn`)이라, 이 셀을 실행하면 SFT 버전이 덮어써집니다.

### 핵심 코드 ⑥ `compute_logprobs` (★★ 빈칸 출제)

In [ ]:
import torch.nn.functional as F

def compute_logprobs(logits, labels, selection_mask=None):
    labels = labels[:, 1:].clone()
    logits = logits[:, :-1, :]

    log_probs = F.log_softmax(logits, dim=-1)

    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)

**이 함수가 하는 일**: "모델이 이 문장(응답)을 생성할 로그 확률은 얼마인가?" → 문장마다 숫자 하나 `(B,)`

**shape 흐름** (B=배치, L=길이, V=50257)

```
logits                   (B, L, V)     모델 출력 (입력 = chosen 전체)
labels                   (B, L)        정답 = 같은 chosen 토큰
labels[:, 1:]            (B, L-1)      위치 t의 정답은 t+1번째 토큰 → 한 칸 앞당김
logits[:, :-1, :]        (B, L-1, V)   마지막 위치 예측은 정답이 없으므로 버림
log_softmax(dim=-1)      (B, L-1, V)   각 위치에서 V개 단어의 로그 확률
labels.unsqueeze(-1)     (B, L-1, 1)   gather의 index는 input과 차원 수가 같아야 함
gather(dim=-1)           (B, L-1, 1)   각 위치에서 정답 토큰의 로그 확률만 뽑음
.squeeze(-1)             (B, L-1)
× mask, sum / mask.sum   (B,)          응답 부분 토큰들의 로그 확률 평균
```

**`torch.gather` 예시** — "각 행에서 index가 가리키는 칸만 꺼내기"

In [ ]:
log_probs = torch.tensor([[[-0.1, -2.5, -3.0],     # 위치 0: 단어 0,1,2의 로그 확률
                           [-1.2, -0.4, -2.0]]])   # 위치 1
labels = torch.tensor([[0, 1]])                    # 위치 0의 정답 = 단어 0, 위치 1의 정답 = 단어 1

picked = torch.gather(log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
print(picked)     # tensor([[-0.1000, -0.4000]])

- `F.cross_entropy`가 내부에서 하는 일(정답 위치의 log 확률 꺼내기)을 직접 구현한 것. 단 부호(−)와 평균 방식이 다름
- `log_softmax` = `log(softmax(x))`를 한 번에 안정적으로 계산
- 빈칸 정답: `labels[:, 1:]`, `logits[:, :-1, :]`, `input=log_probs` (**logits가 아님**), `index=labels.unsqueeze(-1)`
- mask도 labels처럼 `[:, 1:]`로 한 칸 shift해야 위치가 맞음

### 핵심 코드 ⑦ `compute_dpo_loss` · `compute_dpo_loss_batch` (★★ 빈칸 출제)

In [ ]:
def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs,
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    logits = model_logratios - reference_logratios

    losses = -F.logsigmoid(beta * logits)

    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()


def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )

    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )

    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

**숫자로 계산해 보기** (β = 0.1)

| | chosen 로그 확률 | rejected 로그 확률 | chosen − rejected |
| --- | --- | --- | --- |
| policy | −1.0 | −2.0 | **1.0** |
| reference | −1.2 | −1.8 | **0.6** |

```
logits = 1.0 − 0.6 = 0.4                    policy가 reference보다 chosen 쪽으로 0.4만큼 더 기울었음
loss   = −logsigmoid(0.1 × 0.4) = −log(sigmoid(0.04)) ≈ 0.673      (0.693보다 작아짐 = 좋아짐)
chosen_reward   = −1.0 − (−1.2) = +0.2     chosen 확률이 올라감
rejected_reward = −2.0 − (−1.8) = −0.2     rejected 확률이 내려감
margin = 0.2 − (−0.2) = 0.4                클수록 좋음 (교안 출력의 "Train Margin")
```

In [ ]:
# 위 계산을 직접 실행
loss, cr, rr = compute_dpo_loss(
    torch.tensor([-1.0]), torch.tensor([-2.0]),
    torch.tensor([-1.2]), torch.tensor([-1.8]), beta=0.1)
print(loss, cr, rr)        # ≈ 0.6733, 0.2, -0.2

# 시작 시점: policy == reference → loss = ln 2
loss0, _, _ = compute_dpo_loss(torch.tensor([-1.0]), torch.tensor([-2.0]),
                               torch.tensor([-1.0]), torch.tensor([-2.0]))
print(loss0)               # ≈ 0.6931

빈칸 정답 위치:

| 빈칸 | 정답 |
| --- | --- |
| policy 비율 | `model_chosen_logprobs - model_rejected_logprobs` |
| reference 비율 | `reference_chosen_logprobs - reference_rejected_logprobs` |
| logits | `model_logratios - reference_logratios` |
| loss | `-F.logsigmoid(beta * logits)` |

포인트:
- 순서는 항상 **chosen − rejected**, **policy − reference**
- `F.logsigmoid(x)` = `log(sigmoid(x))`를 안정적으로 계산. 앞에 `−`를 붙여 loss로 만듦 (Cross Entropy와 같은 원리)
- `.detach()`: reward는 **기록용**이라 그래프에서 분리 (역전파에 쓰지 않음)
- reference 계산은 `torch.no_grad()` 안에서 → gradient 불필요, 메모리 절약
- 모델 forward가 배치당 **4번** (policy×2, reference×2)

### 핵심 코드 ⑧ `train_model_dpo_simple` · DPO 실행 설정

In [ ]:
def train_model_dpo_simple(policy_model, reference_model, train_loader, val_loader,
                           optimizer, num_epochs, beta, eval_freq, eval_iter, start_context, tokenizer):
    tracking = {"train_losses": [], "val_losses": [], "tokens_seen": []}
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        policy_model.train()
        for batch in train_loader:
            optimizer.zero_grad()

            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch, policy_model=policy_model, reference_model=reference_model, beta=beta
            )

            loss.backward()
            optimizer.step()

            tokens_seen += batch["chosen"].numel()
            global_step += 1

            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model, reference_model, train_loader, val_loader, beta, eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["tokens_seen"].append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                      f"Train Margin {res['train_chosen_reward']-res['train_rejected_reward']:.3f}")

        generate_and_print_sample(model=policy_model, tokenizer=tokenizer, device=loss.device, start_context=start_context)

    return tracking

In [ ]:
# 같은 가중치로 두 모델 생성
policy_model = GPTModel(BASE_CONFIG)
policy_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
policy_model.to(device)

reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
reference_model.to(device)
reference_model.eval()

# optimizer에는 policy_model의 파라미터만
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)
tracking = train_model_dpo_simple(
    policy_model=policy_model, reference_model=reference_model,
    train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, num_epochs=1, beta=0.1,
    eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[2]), tokenizer=tokenizer
)

- 학습 4줄은 그대로: `zero_grad → loss → backward → step`. loss 계산 함수만 `compute_dpo_loss_batch`로 바뀜
- `batch`가 튜플이 아니라 **딕셔너리**라서 `for batch in train_loader:` 로 받고 `batch["chosen"]`으로 접근
- `lr=5e-6`: SFT(5e-5)보다 10배 작게 → reference에서 조금씩만 이동
- `map_location="cpu"`: GPU에서 저장된 가중치도 일단 CPU로 불러온 뒤 `.to(device)`

### Chapter 5 · 6 · 7 한눈에 비교

| | Ch5 사전학습 | Ch6 분류 | Ch7 SFT | Ch7 DPO |
| --- | --- | --- | --- | --- |
| 데이터 | 원문 텍스트 | 문장 + 0/1 | 지시 + 응답 | 지시 + chosen + rejected |
| Dataset | `GPTDatasetV1` | `SpamDataset` | `InstructionDataset` | `PreferenceDataset` |
| 패딩 | 없음 (슬라이딩 윈도우) | Dataset에서 전체 최대 길이로 | collate에서 배치 최대 길이로 | collate에서 배치 최대 길이로 |
| out_head | 768 → 50257 | 768 → **2** | 768 → 50257 | 768 → 50257 |
| 사용하는 출력 위치 | 모든 위치 | 마지막 `[:, -1, :]` | 모든 위치 | 응답 부분 (mask) |
| Loss | `cross_entropy` | `cross_entropy` | `cross_entropy` (`-100` 무시) | `-logsigmoid(β·logits)` |
| 학습 함수 | `train_model_simple` | `train_classifier_simple` | `train_model_simple` | `train_model_dpo_simple` |
| 모델 | GPT-2 124M (무작위 초기화) | GPT-2 124M | GPT-2 355M | GPT-2 355M × 2 (policy, reference) |
| lr | 5e-4 | 5e-5 (LoRA 8e-4) | 5e-5 | 5e-6 |

## 부록. 전체 Tensor shape 한눈에 보기

GPT-2 Small 기준 (B=배치, T=토큰 수, d=768, H=12, hd=64, V=50257)

| 단계 | 코드 | shape |
| --- | --- | --- |
| 텍스트 → ID | `tokenizer.encode(text)` | `list[int]` 길이 T |
| 배치 차원 추가 | `torch.tensor(ids).unsqueeze(0)` | `(1, T)` |
| DataLoader 출력 | `inputs, targets` | `(B, T)`, `(B, T)` |
| 토큰 임베딩 | `tok_emb(in_idx)` | `(B, T, d)` |
| 위치 임베딩 | `pos_emb(torch.arange(T))` | `(T, d)` |
| 합 | `tok + pos` | `(B, T, d)` |
| Q/K/V | `W_query(x)` | `(B, T, d)` |
| head 분리 | `.view(B, T, H, hd).transpose(1, 2)` | `(B, H, T, hd)` |
| attention score | `Q @ K.transpose(2, 3)` | `(B, H, T, T)` |
| context | `weights @ V` | `(B, H, T, hd)` |
| head 합치기 | `.transpose(1, 2).contiguous().view(B, T, d)` | `(B, T, d)` |
| FFN 내부 | `Linear(d, 4d)` | `(B, T, 4d)` → `(B, T, d)` |
| TransformerBlock 출력 | | `(B, T, d)` (입력과 같음) |
| logits | `out_head(final_norm(x))` | `(B, T, V)` |
| 다음 토큰 logits | `logits[:, -1, :]` | `(B, V)` |
| 다음 토큰 | `argmax(..., dim=-1, keepdim=True)` | `(B, 1)` |
| loss 입력 | `logits.flatten(0, 1)`, `target.flatten()` | `(B*T, V)`, `(B*T,)` |

**Chapter 6~7 추가**

| 단계 | 코드 | shape |
| --- | --- | --- |
| 분류 입력 (패딩 후) | `SpamDataset` → DataLoader | `(B, max_length)`, 라벨 `(B,)` |
| 분류 logits | `model(x)` (out_head 768→2) | `(B, T, 2)` |
| 마지막 토큰 | `model(x)[:, -1, :]` | `(B, 2)` |
| 예측 라벨 | `torch.argmax(logits, dim=-1)` | `(B,)` |
| LoRA 압축 | `x @ A` | `(B, T, in)` → `(B, T, rank)` |
| LoRA 복원 | `@ B` | `(B, T, rank)` → `(B, T, out)` |
| SFT 배치 | `custom_collate_fn` | inputs `(B, L)`, targets `(B, L)` (패딩 자리 -100) |
| DPO 배치 | `batch["chosen"]`, `batch["chosen_mask"]` | `(B, L)`, `(B, L)` bool |
| DPO shift | `labels[:, 1:]`, `logits[:, :-1, :]` | `(B, L-1)`, `(B, L-1, V)` |
| gather | `torch.gather(log_probs, -1, labels.unsqueeze(-1)).squeeze(-1)` | `(B, L-1)` |
| 문장별 로그 확률 | `sum(-1) / mask.sum(-1)` | `(B,)` |

## 부록. 교안 빈칸 전체 정답표 (Chapter 2~7)

교안 노트북의 `# TODO` + `????` 빈칸을 **출제된 모양 그대로** 모두 모았습니다 (힌트 줄을 제외한 빈칸 줄 81개. Ch3의 W_query/W_key/W_value 3줄은 한 행으로 묶어 79행). 왼쪽 열을 가리고 정답을 말할 수 있는지 점검하세요.

- 표기: `????` 하나가 빈칸 하나. 정답이 여러 개면 왼쪽부터 순서대로 `①, ②, ...`
- 출처 파일: `1. 언어 모델 이해/Chapter_*_Ex*.ipynb`

### Chapter 2 — Dataset

| # | 함수 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 1 | tiktoken | `integers = tokenizer.????(text, allowed_special={...})` | `encode` | 문자열 → 토큰 ID 리스트. `allowed_special`이 없으면 `<\|endoftext\|>`에서 에러 |
| 2 | tiktoken | `strings = tokenizer.????(integers)` | `decode` | 토큰 ID 리스트 → 문자열 (encode의 반대) |
| 3 | x/y 만들기 | `y = enc_sample[????:context_size+????]` | ① `1` ② `1` | y는 x를 **한 칸 오른쪽으로 민 것**. x가 `[:4]`면 y는 `[1:5]` |
| 4 | `GPTDatasetV1.__init__` | `input_chunk = token_ids[???? : ???? + max_length]` | ① `i` ② `i` | 시작 위치 i부터 max_length개 |
| 5 | `GPTDatasetV1.__init__` | `target_chunk = token_ids[???? + ???? : ???? + max_length + ????]` | `i`, `1`, `i`, `1` | input 구간의 시작과 끝을 둘 다 +1 |
| 6 | `GPTDatasetV1.__getitem__` | `return self.input_ids[????], self.target_ids[????]` | ① `idx` ② `idx` | 같은 번호의 (입력, 정답) 쌍을 반환. 인자 이름이 `index`가 아니라 `idx` |
| 7 | `create_dataloader_v1` | `DataLoader(????, batch_size=..., ...)` | `dataset` | 바로 위에서 만든 `GPTDatasetV1` 객체가 첫 인자 |
| 8 | 임베딩 | `torch.nn.Embedding(????, ????)` | ① `vocab_size` ② `output_dim` | (개수, 벡터 길이) 순서. weight shape도 `(vocab_size, output_dim)` |
| 9 | 임베딩 | `token_embeddings = embedding_layer(????)` | `input_ids` | 토큰 ID 텐서를 넣으면 각 ID의 행을 꺼냄 → `(4,)` → `(4, 3)` |

### Chapter 3 — CausalAttention

| # | 위치 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 10 | `__init__` | `self.W_query = nn.????(????, ????, bias=qkv_bias)` | ① `Linear` ② `d_in` ③ `d_out` | 입력 벡터 길이 d_in → Q 벡터 길이 d_out. W_key, W_value도 같은 모양 (빈칸 3줄) |
| 11 | `forward` | `keys = self.????(????)` | ① `W_key` ② `x` | `(b, T, d_in)` → `(b, T, d_out)` |
| 12 | `forward` | `queries = self.????(????)` | ① `W_query` ② `x` | |
| 13 | `forward` | `values = self.????(????)` | ① `W_value` ② `x` | |
| 14 | `forward` | `attn_scores = ???? @ ????.transpose(1, 2)` | ① `queries` ② `keys` | **Q @ Kᵀ** (순서 주의). `(b,T,d_out) @ (b,d_out,T)` → `(b,T,T)`. 3D라서 `transpose(1, 2)` |
| 15 | `forward` | `masked_fill_(mask..., ????)` | `-torch.inf` | softmax 후 0이 되게 하려면 0이 아니라 **−∞** |
| 16 | `forward` | `attn_weights = torch.????(attn_scores / keys.shape[-1]**0.5, dim=-1)` | `softmax` | 각 행(Query)의 합이 1이 되도록 `dim=-1` |
| 17 | `forward` | `context_vec = attn_weights @ ????` | `values` | 가중치로 **V**를 섞음 (K가 아님) |
| 18 | 실행 | `context_vecs = ????(batch)` | `ca` | 위에서 만든 `ca = CausalAttention(...)` 객체를 호출 → `forward` 실행 |

### Chapter 4 — GPT

| # | 위치 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 19 | `LayerNorm.forward` | `norm_x = (x - ????) / torch.sqrt(???? + self.eps)` | ① `mean` ② `var` | 표준화 = (x − 평균) / √(분산 + ε). 표준편차가 아니라 **var에 eps를 더하고 sqrt** |
| 20 | `LayerNorm.forward` | `return self.???? * norm_x + self.????` | ① `scale` ② `shift` | γ(곱하기, ones로 초기화) · β(더하기, zeros로 초기화) |
| 21 | `FeedForward` | `nn.Linear(cfg["emb_dim"], ???? * cfg["emb_dim"])` | `4` | 768 → 3072로 **4배 확장** |
| 22 | `FeedForward` | `????(),` | `GELU` | 직접 구현한 GELU 클래스 (괄호로 객체 생성) |
| 23 | `FeedForward` | `nn.Linear(???? * cfg["emb_dim"], cfg["emb_dim"])` | `4` | 3072 → 768로 원래 크기 복귀 |
| 24 | `TransformerBlock.forward` | 첫 번째 `x = self.????(x)` | `att` | norm1 → **att** → dropout → + shortcut |
| 25 | `TransformerBlock.forward` | 두 번째 `x = self.????(x)` | `ff` | norm2 → **ff** → dropout → + shortcut |
| 26 | `GPTModel.__init__` | `self.tok_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])` | `vocab_size` | 토큰 종류 수만큼 행 |
| 27 | `GPTModel.__init__` | `self.pos_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])` | `context_length` | 위치 개수(최대 길이)만큼 행 |
| 28 | `GPTModel.__init__` | `self.out_head = nn.Linear(cfg["emb_dim"], cfg["????"], bias=False)` | `vocab_size` | 768 → 50257: 단어마다 점수 |
| 29 | `GPTModel.forward` | `x = tok_embeds + ????` | `pos_embeds` | 토큰 임베딩 + 위치 임베딩 (broadcast) |
| 30 | `GPTModel.forward` | `x = self.????(x)` (dropout 다음) | `trf_blocks` | `nn.Sequential`로 묶은 12개 블록을 한 번에 통과 |
| 31 | `GPTModel.forward` | `logits = self.????(x)` | `out_head` | final_norm 다음 마지막 층 |
| 32 | `generate_text_simple` | `with torch.????():` | `no_grad` | 생성(추론)에는 gradient 불필요 |
| 33 | `generate_text_simple` | `logits = logits[:, ????, :]` | `-1` | **마지막 위치**의 예측 = 다음 토큰 |
| 34 | `generate_text_simple` | `idx_next = torch.????(logits, dim=-1, keepdim=True)` | `argmax` | greedy: 점수가 가장 큰 토큰. keepdim으로 `(B, 1)` 유지 |
| 35 | `generate_text_simple` | `idx = torch.????((idx, idx_next), dim=1)` | `cat` | 생성한 토큰을 시퀀스 뒤(dim=1)에 이어 붙임. 인자가 **튜플** |

### Chapter 5 — Pretraining

| # | 위치 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 36 | `calc_loss_batch` | `loss = torch.nn.functional.????(` | `cross_entropy` | 다음 토큰 분류 문제의 loss. logits를 그대로 넣음 (softmax 안 함) |
| 37 | `calc_loss_batch` | `logits.????(0, 1),` | `flatten` | `(B, T, V)` → `(B*T, V)` |
| 38 | `calc_loss_batch` | `target_batch.????()` | `flatten` | `(B, T)` → `(B*T,)`. 인자 없음 |
| 39 | `train_model_simple` | `model.????()` (epoch 시작) | `train` | dropout 켜기. 평가 함수가 `eval()`로 바꿔 놓을 수 있으므로 매 epoch 다시 설정 |
| 40 | `train_model_simple` | `optimizer.????()` (첫 줄) | `zero_grad` | 이전 배치의 gradient 초기화 (안 하면 누적) |
| 41 | `train_model_simple` | `loss = ????(input_batch, target_batch, model, device)` | `calc_loss_batch` | 배치 하나의 loss (`calc_loss_loader`가 아님) |
| 42 | `train_model_simple` | `loss.????()` | `backward` | 역전파: 모든 파라미터의 gradient 계산 |
| 43 | `train_model_simple` | `optimizer.????()` (마지막 줄) | `step` | 계산된 gradient로 가중치 갱신. **zero_grad → loss → backward → step** |
| 44 | `generate` | `logits = logits[:, ????, :]` | `-1` | Ch4와 같음 |
| 45 | `generate` | `idx_next = torch.????(logits, dim=-1, keepdim=True)` | `argmax` | Ch4와 같음 |
| 46 | `generate` | `idx = torch.cat((idx, ????), dim=1)` | `idx_next` | 이번엔 함수 이름이 아니라 **붙일 대상**이 빈칸 |
| 47 | `text_to_token_ids` | `encoded_tensor = torch.tensor(encoded).????(0)` | `unsqueeze` | `(T,)` → `(1, T)` 배치 차원 **추가** |
| 48 | `token_ids_to_text` | `flat = token_ids.????(0)` | `squeeze` | `(1, T)` → `(T,)` 배치 차원 **제거** |
| 49 | `main` optimizer | `AdamW(model.????(), ????=settings["learning_rate"], ...)` | ① `parameters` ② `lr` | `model.parameters()`는 **메서드 호출**(괄호). 학습률 키워드는 `lr` |

### Chapter 6 — 분류 미세 튜닝 · LoRA

| # | 위치 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 50 | `SpamDataset` | `encoded_text[:????]` | `self.max_length` | 긴 문장은 앞에서부터 max_length개만 (`max_length` 지역 변수가 아니라 `self.`) |
| 51 | `SpamDataset` | `encoded_text + [????] * (self.max_length - len(encoded_text))` | `pad_token_id` | 50256으로 남는 길이만큼 채움 |
| 52 | `SpamDataset.__getitem__` | `label = self.data.iloc[index]["????"]` | `Label` | CSV 컬럼 이름 (대문자 L). 텍스트 컬럼은 `Text` |
| 53 | `calc_loss_batch` | `logits = model(input_batch)[:, ????, :]` | `-1` | 문장 전체를 본 마지막 토큰의 출력만 분류에 사용 |
| 54 | 모델 수정 | `param.requires_grad = ????` (전체) | `False` | 전체 동결 |
| 55 | 모델 수정 | `torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=????)` | `num_classes` | 768 → 2 |
| 56 | 모델 수정 | `for param in model.trf_blocks[????].parameters():` | `-1` | 마지막 Transformer 블록 |
| 57 | 모델 수정 | `param.requires_grad = ????` (마지막 블록) | `True` | 동결 해제 |
| 58 | 모델 수정 | `param.requires_grad = ????` (final_norm) | `True` | 동결 해제 |
| 59 | `LoRALayer` | `self.A = nn.Parameter(torch.empty(in_dim, ????))` | `rank` | A: (입력 → 좁은 통로) |
| 60 | `LoRALayer` | `self.B = nn.Parameter(torch.zeros(????, out_dim))` | `rank` | B: (좁은 통로 → 출력), **0으로 초기화** |
| 61 | `LoRALayer.forward` | `(self.???? / self.rank) * (x @ self.???? @ self.????)` | ① `alpha` ② `A` ③ `B` | 배율 alpha/rank. **A 먼저, B 나중** |
| 62 | `LinearWithLoRA.forward` | `return self.linear(x) + self.????(x)` | `lora` | 고정된 원래 출력 + LoRA 변화량 |
| 63 | `replace_linear_with_lora` | `if isinstance(module, torch.nn.????):` | `Linear` | Linear만 LoRA 버전으로 교체, 나머지는 재귀 |
| 64 | LoRA main | `param.requires_grad = ????` | `False` | LoRA 주입 **전에** 전체 동결 |
| 65 | LoRA main | `????(model, rank=LORA_RANK, alpha=LORA_ALPHA)` | `replace_linear_with_lora` | 동결 후 주입 → 새로 생긴 A, B만 학습 |

### Chapter 7 — 지시 미세 튜닝 · DPO

| # | 위치 | 교안 빈칸 | 정답 | 이유 / 헷갈리는 점 |
| --- | --- | --- | --- | --- |
| 66 | SFT collate | `new_item += [????]` | `pad_token_id` | 끝에 50256 하나를 붙여 "응답 끝"을 학습 |
| 67 | SFT collate | `padded = new_item + [????] * (batch_max_length - len(new_item))` | `pad_token_id` | 배치 최대 길이까지 채움 |
| 68 | SFT collate | `inputs = torch.tensor(padded[:????])` | `-1` | 마지막 토큰 제외 |
| 69 | SFT collate | `targets = torch.tensor(padded[????:])` | `1` | 첫 토큰 제외 = 한 칸 shift |
| 70 | SFT collate | `targets[indices[????:]] = ignore_index` | `1` | **첫 번째 패딩(인덱스 0)은 남기고** 두 번째부터 −100 |
| 71 | `PreferenceDataset` | `tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")` (chosen) | `chosen_response` | |
| 72 | `PreferenceDataset` | `tokenizer.encode(f"{prompt}\n\n### Response:\n{????}")` (rejected) | `rejected_response` | |
| 73 | `compute_logprobs` | `labels = labels[:, ????:].clone()` | `1` | 정답은 한 칸 앞당김 |
| 74 | `compute_logprobs` | `logits = logits[:, :????, :]` | `-1` | 마지막 위치의 예측은 정답이 없어서 버림 |
| 75 | `compute_logprobs` | `torch.gather(input=????, dim=-1, index=labels.unsqueeze(-1))` | `log_probs` | raw logits가 아니라 **log_softmax 결과**에서 꺼냄 |
| 76 | `compute_dpo_loss` | `model_logratios = ???? - model_rejected_logprobs` | `model_chosen_logprobs` | chosen − rejected |
| 77 | `compute_dpo_loss` | `reference_logratios = ???? - reference_rejected_logprobs` | `reference_chosen_logprobs` | chosen − rejected |
| 78 | `compute_dpo_loss` | `logits = model_logratios - ????` | `reference_logratios` | policy − reference |
| 79 | `compute_dpo_loss` | `losses = -F.????(beta * logits)` | `logsigmoid` | −log σ(β·logits). 처음엔 logits=0 → loss = ln 2 ≈ 0.693 |

### 빈칸 유형별 패턴 (외우는 요령)

| 유형 | 빈칸 예 | 공통 규칙 |
| --- | --- | --- |
| shift (한 칸 밀기) | Ch2 #3, #5 · Ch7 #68, #69, #73, #74 | 입력은 `[:-1]` 또는 `[i : i+n]`, 정답은 `[1:]` 또는 `[i+1 : i+n+1]` |
| 마지막 위치 | Ch4 #33 · Ch5 #44 · Ch6 #53, #56 | `-1` (`[:, -1, :]`, `trf_blocks[-1]`) |
| 패딩 | Ch6 #51 · Ch7 #66, #67 | `[pad_token_id] * (최대길이 - len(...))` |
| 학습 루프 | Ch5 #39~43 | `train() → zero_grad() → calc_loss_batch → backward() → step()` |
| 차원 추가/제거 | Ch5 #47, #48 | 추가 `unsqueeze(0)`, 제거 `squeeze(0)` |
| Embedding/Linear 인자 | Ch2 #8 · Ch3 #10 · Ch4 #26~28 · Ch6 #55 | Embedding(개수, 차원), Linear(입력 길이, 출력 길이) |
| 동결 | Ch6 #54, #57, #58, #64 | 전체 `False` → 학습할 부분만 `True` |
| 순서가 있는 뺄셈 | Ch7 #76~78 | chosen − rejected, policy − reference |

## 부록. 빈칸 대비 체크리스트

시험에서는 아무 줄이나 비워질 수 있으므로, 아래 항목을 **보지 않고 쓸 수 있는지** 점검합니다.

### 함수/클래스 통째로 쓰기

- [ ] `GPTDatasetV1` (`__init__`, `__len__`, `__getitem__`)
- [ ] `create_dataloader_v1`
- [ ] `CausalAttention`
- [ ] `MultiHeadAttentionWrapper`
- [ ] `MultiHeadAttention` (view/transpose/contiguous 순서 포함)
- [ ] `LayerNorm`, `GELU`, `FeedForward`
- [ ] `TransformerBlock`
- [ ] `GPTModel`
- [ ] `generate_text_simple` / `generate`
- [ ] `calc_loss_batch`, `calc_loss_loader`, `evaluate_model`
- [ ] `train_model_simple`
- [ ] `text_to_token_ids`, `token_ids_to_text`, `generate_and_print_sample`
- [ ] `main`의 optimizer · train/val loader 부분
- [ ] `create_balanced_dataset`, `random_split`
- [ ] `SpamDataset` (자르기 · 패딩 · Label)
- [ ] `calc_accuracy_loader`, 분류용 `calc_loss_batch` (`[:, -1, :]`)
- [ ] `train_classifier_simple`
- [ ] freeze → out_head 교체 → 마지막 블록/final_norm unfreeze
- [ ] `LoRALayer`, `LinearWithLoRA`, `replace_linear_with_lora`
- [ ] `format_input`, `InstructionDataset`
- [ ] SFT `custom_collate_fn` (+ `partial`)
- [ ] `PreferenceDataset`, DPO `custom_collate_fn`
- [ ] `compute_logprobs`, `compute_dpo_loss`, `compute_dpo_loss_batch`
- [ ] `train_model_dpo_simple`

### 헷갈리기 쉬운 한 줄

| 위치 | 정답 |
| --- | --- |
| tiktoken 인코딩 | `tokenizer.encode(text, allowed_special={"<\|endoftext\|>"})` |
| target 슬라이싱 | `token_ids[i + 1 : i + max_length + 1]` |
| 루프 범위 | `range(0, len(token_ids) - max_length, stride)` |
| 위치 임베딩 입력 | `torch.arange(seq_len, device=in_idx.device)` |
| 마스크 생성 | `torch.triu(torch.ones(context_length, context_length), diagonal=1)` |
| 마스크 적용 | `attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)` |
| scale + softmax | `torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)` |
| 단일 head 전치 | `keys.transpose(1, 2)` |
| MHA 전치 | `keys.transpose(2, 3)` |
| head 합치기 | `context_vec.contiguous().view(b, num_tokens, self.d_out)` |
| LayerNorm 분산 | `x.var(dim=-1, keepdim=True, unbiased=False)` |
| LayerNorm 출력 | `self.scale * norm_x + self.shift` |
| FFN | `Linear(emb, 4*emb) → GELU() → Linear(4*emb, emb)` |
| 블록 쌓기 | `nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])` |
| out_head | `nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)` |
| 마지막 토큰 | `logits[:, -1, :]` |
| greedy | `torch.argmax(logits, dim=-1, keepdim=True)` |
| 이어 붙이기 | `torch.cat((idx, idx_next), dim=1)` |
| loss | `F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())` |
| optimizer | `torch.optim.AdamW(model.parameters(), lr=..., weight_decay=...)` |
| 학습 4줄 | `zero_grad() → loss 계산 → backward() → step()` |
| 평가 3종 | `model.eval()` + `with torch.no_grad():` + 끝나면 `model.train()` |
| 배치 차원 | `unsqueeze(0)` 추가 / `squeeze(0)` 제거 |
| 저장/로드 | `torch.save(model.state_dict(), path)` / `model.load_state_dict(torch.load(path, weights_only=True))` |
| 균형 맞추기 | `df[df["Label"] == "ham"].sample(num_spam, random_state=123)` |
| 섞기 | `df.sample(frac=1, random_state=123).reset_index(drop=True)` |
| 분류 패딩 | `encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))` |
| 분류 라벨 | `self.data.iloc[index]["Label"]` |
| 분류 logits | `model(input_batch)[:, -1, :]` |
| 분류 loss | `F.cross_entropy(logits, target_batch)` (flatten 없음) |
| 정답 개수 | `(predicted_labels == target_batch).sum().item()` |
| 동결 | `param.requires_grad = False` |
| 분류 헤드 | `torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)` |
| 마지막 블록 | `model.trf_blocks[-1].parameters()` |
| LoRA A | `nn.Parameter(torch.empty(in_dim, rank))` + `nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))` |
| LoRA B | `nn.Parameter(torch.zeros(rank, out_dim))` |
| LoRA forward | `(self.alpha / self.rank) * (x @ self.A @ self.B)` |
| LoRA 합 | `self.linear(x) + self.lora(x)` |
| LoRA 교체 | `setattr(model, name, LinearWithLoRA(module, rank, alpha))` |
| Input 섹션 | `f"

### Input:
{entry['input']}" if entry["input"] else ""` |
| Response 섹션 | `f"

### Response:
{entry['output']}"` |
| 배치 최대 길이 | `max(len(item)+1 for item in batch)` |
| SFT 입력/정답 | `padded[:-1]` / `padded[1:]` |
| SFT 마스킹 | `targets[indices[1:]] = ignore_index` |
| collate 고정 | `partial(custom_collate_fn, device=device, allowed_max_length=1024)` |
| 응답만 추출 | `generated_text[len(input_text):].replace("### Response:", "").strip()` |
| DPO 프롬프트 마스크 | `mask[:prompt.shape[0] + 2] = False` |
| DPO shift | `labels[:, 1:].clone()` / `logits[:, :-1, :]` |
| gather | `torch.gather(input=log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)` |
| DPO logits | `(model_chosen - model_rejected) - (reference_chosen - reference_rejected)` |
| DPO loss | `-F.logsigmoid(beta * logits)` |